# SPY · 42-candle images · GitHub storage

**Repository: [jamelof23/SPY](https://github.com/jamelof23/SPY)**

This notebook downloads the images from GitHub Releases and saves the completed
trained model there. **AUTO mode loads the selected saved model and skips training.**
If that model version does not exist, it trains once and publishes it.

The dataset remains **42 daily candles, EMA 8, EMA 21, and volume**, with image pixels
as the model's only features. Train on **2020–2024**, develop on **2025**, and reserve
**2026 through November 20** for the explicitly disabled final test.

| When | What happens |
|---|---|
| First setup, no dataset release | Recreate 1,733 images from the exact historical snapshot bundled below and upload the ZIP to a dataset release |
| First model version | Train, save the weights/calibration/settings, and upload a model release |
| Later AUTO runs | Download the dataset and saved model as needed, then predict without retraining |
| Deliberate new experiment | Choose a new `MODEL_TAG`, such as `spy42-model-v2` |

**No image ZIP upload to Colab is required.** This single notebook contains the small
source-history snapshot and code needed for first setup. Its embedded source is the
same downloaded OHLCV already used for the approved charts. GitHub Releases hold the
large ZIP/model files; ordinary repository code files stay small.

This is a fixed historical snapshot ending **November 20, 2026**. It does not fetch new
daily prices automatically. The model forecasts next-session open-to-close direction
and candle body; conditional intraday EMA-entry discovery is a later stage.


## One-time setup for saving to GitHub

1. In GitHub, create a **fine-grained personal access token** for resource owner
   `jamelof23`, repository **SPY only**, with **Contents → Read and write** and an expiration.
2. In Colab's left sidebar, open **Secrets** (key icon), add a secret named
   **`GITHUB_TOKEN`**, paste the token there, and enable this notebook's access.
3. Select a GPU runtime if available, then run the cells in order.

Public dataset/model downloads work without a token once the releases exist. First
setup and new-model uploads require the secret. Never paste a token into notebook code
or output. The code keeps it only in memory and never includes it in saved artifacts.

The repository is currently public, so published datasets and model releases will be
public too. A completed model is uploaded automatically; a run interrupted before
completion is not an automatically resumable GitHub training checkpoint.

References: [GitHub Releases](https://docs.github.com/en/repositories/releasing-projects-on-github/about-releases)
· [Fine-grained tokens](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens)


In [ ]:
# Configuration — the defaults use your repository and automatically reuse model v1.
from pathlib import Path

GITHUB_REPO = "jamelof23/SPY"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"
DATASET_TAG = "spy42-data-2026-11-22-v1"
MODEL_TAG = "spy42-model-v1"
MODE = "auto"  # auto: load saved model or train once; predict: never train; train: new tag only

BOOTSTRAP_DATASET_IF_MISSING = True
WORK_ROOT = Path("/content/spy42_github")
EPOCHS = 15
BATCH_SIZE = 16
NUM_WORKERS = 2
RENDER_WORKERS = 2
USE_PRETRAINED_WEIGHTS = True
FINE_TUNE_LAYER4 = False
RUN_SMOKE_TEST_BEFORE_NEW_TRAINING = True

# Development-only, signed-SPY return proxy; never options PnL.
PROXY_ROUND_TRIP_COST_BPS = 0.0  # Explicit frictionless default; change before final testing.
MIN_VALIDATION_SIGNALS = 30
RUN_FINAL_TEST = False  # Keep False while choosing settings. Normal Run All skips 2026 assessment.


In [ ]:
# Check the standard Colab runtime; no silent package upgrades.
import importlib.util
import sys
import subprocess
import json
import hashlib

required = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy",
            "pandas": "pandas", "PIL": "Pillow", "matplotlib": "matplotlib",
            "requests": "requests"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError("Missing packages: " + ", ".join(missing) +
                       ". Use a standard Colab runtime with compatible torch/torchvision.")
import torch
import torchvision
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Mode:", MODE, "Model version:", MODEL_TAG)


## Load the bundled code and historical snapshot

The following collapsed cell contains compressed text files, including the original
history CSV and the chart renderer/trainer. Each embedded file is checked against its
SHA-256 hash before use. This makes first setup independent of manual image uploads
or changes to the external data provider. It does not contain a trained model or secrets.


In [ ]:
#@title Load bundled code and exact historical source (no manual editing needed)
import base64
import zlib
import importlib

BUNDLED_FILES = {'github_artifacts.py': {'sha256': 'f6151d10b5d75283a08fac3ba013e92cc1df5894de20f3f1efda90c851714618', 'zlib_base64': 'eNrNPGlz48Zy3/UrxnhVMbELgpR2bT9zzXXkXdm7sb2rkmQnsaxigcRQggUCfDgk0Qyr8h/yD/NL0t1zYAYHj83Lq7jKK2IwR09P390Dx3EuF0EceyxaLMoimMac/RAV78opu+AxD3LOgqyI5sGsYHc8XvKMzdOMFXecXZ7/+8sT9iaNgyl7TLP7eZw++kdH5+U0jmYs40GYs4Q/wIgSZsEReblcxhEPWZHe88Rn5zxbRHkepQmb3fHZfe6xMAvmBYuSfMlnBbw4CpKQBXnOCxbMZjyHLuUyToPQY/hmSYsF2JMFJayRFPjIffYhZXH0wAkylpVJzqBLtFimWeEfOY5zNM/SBZtM5mVRZnwyke9g1iQtaML86Ei23QX5XRxN1eMfOcAlfy+iBS9WS56rhjQXMy+DAseoac/h0WPnsNR5mkdP+KhGZFz9ymFh9bvgi+U8irmYrcximMxfBhmgUvb4W5kW3MNXf6RRQj9yQK+e4c9ITFCt87eS5wVs6/T8/eTXs4vL9x8/sDFzToYnX/aHL/rHQ4deXXz8eIXtd0WxzEeDQbCM/NuouCun/ixdOEe/XJ5dTE5/OPtAvYgM+kQG/VNJKv13RCqDYx+mfP/hCrpOfj69+PHsAkd881m/z/LlCoYp2sr7D8cj5+jtx3/98NPH07eTdx8vry6h79oxFvaYkwma7BNF5BIqIK9sliYFHL7ql07/APrp6rA5OjqaxTCHJHUF9lmWpVnvogQiWnB6cEdHDP4DejlNKkbg+OqVSXBALoOEF0Rs9FaR/oIHCfucqDy5/dxHwlNrT86e4LhmUfEm4yHOE8T5xyRe9dRB+biAfwr/fAd7rkD5kfMl43KwOlbgziDksO7jHRw6W2awPMyZ3CK5iJ4+QJjN2KxaTsCD04Z8DswwA0kwmfRyHs89NbFcF//LOPBKol7ARnAU0tgkvwtOvviyh0Q/AjLO2H8Qxbus/xofxRRhdIuAjhU/+XKUS28f4ahoEM3i+umSJz0nmzou8D9OwoNFBQoKodldmdyDrGBRwbNeHCymYTCSPX2UP73j4clL9ozhH9djU8cxNlNB5JfLEGRGj+YTwMidyvd3/En8AlDFpidJsOC9hyAuOa1obzSaMxAiLIIzB5ZOZrKnJ3oC5Pg24/68jONFUMzueplzfdr/Lej/Oex/fVP99Cf9m/XQO/766w1QNU1iHkcQgTT4FVsF5TqKkBmCl7NFCeiegtxFEuBsmpZJCNIXT4w6+I61W5pfbbAIbv9x+zv5ar/9KZUEwHXuDt6xnbsDyjN2B9T6IU242qR81HuljrBbo9WYFxs/HSuAhKA/P+1/f7P+8uV+SACxAbINdhoAzz8B088i0ODs8t1pH7hJ0WzL5n1Q0MAomoZJiBIq6NcIhs6KViRozqWO/i0veo5ok8vA3mWfT0CTGGnjSe4CemRFjqKh5whpMXIayGkT4o40YsTqiKyEgREABgioQq5MilZcEUbkm4wv0gcOwnQePRkQaBSSipzcpSAaQP0SMYFhEiOiQ3jixTW03BBOK6wU2arawhI3CJhV2hvncau3qLjHopOPD/SGP834smC9KzA8aLueQSD7oeeC/6GICIgHpOkCfyvx8cvFT4AURqaHeXA9AUgO1tqCs8+UjeDgqYlXiArkPnHGicKFJXV1Z9TL1Fm3LOFcQImG1IJ7l9P0EAqPvXz5wj10f3juXLOMvcMOOyAfGToxSqLC0InLVJ4y2bCm+IBzIilSwdckdRxfETqIYWpx2Wt2/HLYKiMszFlitPftqHoiOfriq43x3v12UD1NfOhx7B0PhyhkcFE9MYGCLb4gQGfguNf94xvE+9rx0ZzyfWdja86mWPoFXQUthdPHBOw/nDWPijRbkUj2aHuBQr6BJkImyg7sQMhE675Xwx71suWE8CSgJVr2XAvC9v9gWJCsQNUHmQ9G2TKAaV1pTQQZbplmdN1d270Cf0ZArfRQAAAlYLUXK2B8MLIKMmnSskCTrOC0lrltpChfEgT+nIjpxky00ZPdeTJF5Tdmc0dgdrDGPxun1ivnwqcaa5PfvxRNBor+Av4dOVnA/0lwy8O+sl+XWfq0Ivy/OUURhkYkuF1T2IiwaaNkDv+aZmQ7AGS/AhSdhm4NFxPcDfDbPJWsZJimcieSDcHnuktDcnk89syzXL9w/D3Mz7UYHvdOhkMPzL9nz+4fg+w2N462Jr89oPeaswPEL/zN3GzdVJArsxt8FYtinNMZCmlnxGDKpXJSBw9JKCd6jm6k49mD/q0vJFH/dBn1f4V5YQxMYThstQHAdVn/9BY2D90qx6zqtalDqqxdgQxQKsueI1/Bbtcb12JMC7M2T0jpZtDuqMGAWyQ0OFTah5HKWrKNFgZ46FFG8QL2mAEPmexjbOnaQScpzaI/CcnODfHIdzzIgE7XBoAGq1hKWGj/fAlOP/KXTcWK9Bqbs6hQgjKWf4F/wYcE5h/3jr/w2PFfh67XmIAoFEg7hE2C1lGEqwjV6l9tXGp/zdsX4scZNQPAbZKr7QQa4CibSQkC5VnOA3AYQh8EBiCNBXPwthBbINbRvQTHOsFYzUNUrF4xpzlrcQdHKUU7WBvCFWbSk/ftAXWbQ9KZOhu0B4syn8zSsG5h2LsW/UgCNke2Hrs/i9Oc13RIyAvYO8Ys3mB8ypR5Hlvq6BU8oAoFFmFxtIjgTByEWgKB5svL4TFYL8MXLuNwxIy0h0LvYyBwA4IF/BhgsxpOuk9w3rBy311dnQPF08Ibn63FBjYYDZOI1yEXONY4ngawKwSgCFCPGuylfX2BHiGK/xlnjmaC8CvhjJKsp3oa4rWFx6xJfRrYoOxKze5Lyg08RAl4PFHI/uXy44emJUsBhCiB/XfIAE0M1R5BL9SUT17OwSv4X+kfiQ8hbwgZShG2iBy1rg7RPTetgucKHmuMDZj1VAEo/7YIHtfAADh9pKAJRPJo0F2s9uI4DiisaL6SQVlmGH8ZD8C8mkYxyQhYTwbJPDioWVyGKBBeDl964ogpKqW5X5gDYqd4CM4PZ1eolm0DsmYp4ihP+rPAmmAfUpPwXNG4pgCOmMZVfjE6NNooU42Hk6DedUSyAszBECgRASSLvmECWmYP/qmTB7XVzoGEpyTIIritgjS4Z8t3l2dz9oQ8j8GROE3vy+UrKcjlYRH2g1qEns0CZNZZilZfIGLz/pGelex0qaNzeC9PGaOPdQrAuXWjiOGIFZPKNew9AnFCP7E0CiUFjUuQUfgfzg6wFmSzO+RgIpwkFYjF3xJEY9+VMIKtj0VcC/6vnYEmblMA1uyBVqaU1NjFk2io03bzAcasBg60UdweYQAREsz5GEiwxrSm4EA8Ge/dnWpxPMYho/2VnCWECIGEINJghl9CissSoeokx6YA02pgqxdMAw0GVdYA8SesLtiTOBLPrd6DoHTcT1YPdlDAJsxWJUhvKh6URD8ho3QiElNdcnGXmbwF7PMsfQAhAtTfZiBPObitktdyon6dTbG8604BWtdbV1lpH9s/QmyemiAcJj0NA0xKTgGo0e4Qda039dPAGIDRjcbd85WLdiqigSIC0EDGm7Ms8zvc9CKIErCnEvwdhAv44e6/0Ytqa0Q2BviGCZjMIwrFiSQrQCMCHE2atJVCmD4m6KWKmG7PEm6eol8R4PVE7JNOUoayQhBrYA6R82BkbzzDMBOsIvM8LVEv0ZfIH4e2mQV59CcfiLhqTuEFOOw4mAF5F+kiwtQT2AYgzClqADsHFIA1Egr3L2Bv3n6wDAMRUJJSndIx1b7A2iJB3yFWKmQSTXEkn2sREUa8i19RYssckXKEo7++IdlohMKlsALJi79uTGLDEJ9chBjleLS3ba+SHWHKNXUg9cFpQK94xShMRrDisiFb45+NSSvi7Vht83p4Y6uPpkhlOt8umlGzSEEsAyE8dPam+beUzRdQiNF4sGDzSVfnlbIuVkLfF+lyif6PoZ2BaOy8A7bYtp+hWfClx6ZpGusIYeMt8LB+OWTfjMUa8PdEpgqfPXuxv/iivdEMsDPl3MLs6MvwMKe6hxMY+53mUREjNTep2AtYcxlNdLKFEhE11kPKruVtLFxUkW48x2o2fNTv4DCrV3tv9ecgieZqLqmQxNGK4o1ykYOAyYPbjFtKVCwDqm9C/WBjGhBkNg3IUZvTMgF5mBfK9oKxeLa9NsrtDFvZU/yfxbBao1Y141nScS2KJc4TtJxF51FY62ZTuhrUTe1VD0XxeiEg9uFBGzUZWSjmeYSR4Fx52vU9lVlsbwcaal7bfgHYTQMJVqbMx2IB0xKZO+vKAt9U5rcQ34O1wsHGaTdVDkAEGP2YyqgENIlZ4vgIi01Cy6CxGB5s6dF2fE2z9DHn2URr9r0R2I08LUsQaSKPsNaWm4ErteZgXfdVPv/c3ahWoYB1s7P7nPB8LBgOYzZCuoXwKY9TELfAaEUVULOwjm7XACG1JFJl7wAWqObEaHJ9gBFkHGYne5ZUMTr5UT7JV4s4Su57h9iA85IUBPizq8U0BcnRxykq3WAu0Q4xFoFhIdPiPoyynnjIyXb3ALkRSLr0vmbKK5/Mo6qyNAtA30qzzfboGgE7NIcmZAoFyS3vfdnCLAdI28Nc6xY3myi86bR4neNsV/oFRmFfDE/wnxf4z1f4z1/d7vEqqr/uSOekQMpFX5QcORtPFh9tAao9R7lV41E4sTGsKUI6RYJdVefudyQq+3HLtxyLlflYdyaj2H7Y61ymmUqxEN2RR9kT/11xFPT8ugimHfFFkN2Sua3nUyk3kug/pWLbzvYTaI3R1OwVsdKoE2E7pajCFbsL0LnuEjtNDSWrTQVhCTDau6OzEiUl3xvfoBuASTsExhY3SWxIC9BaKqRtqc0rFlVpFyQkURGkzPc2DEyh1/3RbgbcgvmrFAOVyUrVHlUQS7qtr6vqouAljx542FI56bGhNYSKKFXpsP8B3cIrJfK/h6YmK8Pa46Z28ZioebJeUanOc0YlISLZMHawf+FgCAH9uXFTVrVWb2qOMdQRqWFZvEk+fKO3VfOpDxaLPyeyuleUcE7QHRubtZ+jbcxEY7p5qZOQBdPKo3k+JlffKCFtpXvZ+zV5jJ/Ev28l2cAs01UBdpBwNMkACvksDjJd3YZrdHFzZ+Vr/T95IhSyautnbusz6UvDOfXqXh86jc1yWhxS63mgPV6hQwZwKFkNDJ5rWFRlJE1fR0ia+zIO1dPUaEXEWoPlxvu/e15eSjSVhddyQuyMQmRk51G+w7BkUaDJrYQH5DwNeWwWYLWcwjYFhTVcmpm3TqO7+WVC5rOUupXZWiVdRTZCZ7kM0vgLswPHsUQILA12e8jSJFalCoGMmKk4P4YMxTHmmCnxrZryZXDLK3sXtP/Jcd28l15SW0x9XuV7vl2CWMLZQBAN/4l+rPHfzY6w+9aMSe7RRg9jkV+qTIfK2Ehs1XnBiIYCty8IIfSjioXmFIaAtkaqZkypmpv6RqwA6Ot6/HNXyKmMiwiLu/XqoPgyLiPjwS3GD/M0fhDXi4LFNLotMWUAarZESvebIRQJS6sfo314CpVuSTXtk7sRaRlRkJ4zhFrfsHolNiDvUz1GcUxLY6RIpAaAcdoEtxQ+LdFcA9kKVS77BpRfW5inVpC902cVV26QZkToUiR6URTJoNY050CkoCnLOJR+OXvAoH8kgrnbakdIKIuQZl4lBquI6DNPZ9tk7kAa/SqGLGwO17wagjCOzazPflF9G1Vk7YybYf4GpxrVzmZgk9qhjcD5hBgTmHpmkhJmDEvhOsmof46qoKDEH4p+YYGF7UYkRpM7w8aquHtX2F+HEM1AvBnTuab0x7V4c3PUUX/bq52nYRqo2wNu59iqn1UpLJvbYCFL2blxPz3Op8LbA8IKJengHABwYW0jKwscvUKrGXxgKY8kdwB4eAFS1yikWXSLipju4TSMfiIXAbyiPjMEpaiepsVUH/WXOdhCZwvcA1M0tayMea2yIzEjpQgtX2lsmYzuKFCRqcd8RAroGi3+GyyULGKViZym4cquZbEyiW/IVxtUUXW78EPdDdV5GLGch6eRKNiMApYzyjwtTUktAvji7oDPBHJ0jqqqdZEmhZFkl/FedbdqQNlOdNdQGVV5EyUWKxNO0Bj25EYh9QfJ2ZFmbCR/pSFgMbzOqigRX6F1WGb8sNqXpqFBh1ErsccmdV2BHlCRn3zxRUvOAc/Pvt+ALdj/i+FkWNdGWy544TIDHFtLqxXgRGPMt3Z/wL5jClKXhqLAMEtY6E6B2YDl/mp3tXaC290bXv6k87OiFIrEAFBikN0DpdL8WLth1S8A/ZOxZdx6NEPPvjRu5AUFiqBjwSsR9U39EI8xq4MYp2ldSmgO1SmsKU1CKsmcirpuSHpUI3ftWVWlHP/3f/4XrFAmEfg58UomoaUXVhdtWo819LIFS6sKFtl9vYNWW5neRvkEF+5O6WAIBDjrexRaYjeE9dYkiZbBa6HPRmIN1AFwNOCwimyyUjCjxkVYd1O/6tFSNGRiSAqEsVmSJsRonWl137oexAf18tNKpbbbrSVd8kn4IwoyKl2BXna1EcZhCGvwBrBCuSNlZo00djdGCQZwSDghhh0Lvn3OnN+T3xMslrMZ+zndvffDcrHMe2IpkDZpVkzu+UoZhjlfBllQpFk+7jke5szwxh5Oyvr9105L+ZrCWEtxRLcvi/XvFdyWUYltovLIsbNijfyKLrTSc/lgRSdFz9q3qBxR4tbsnMmbTz5oH3lNkja51V8xmLlpaInZYUeEabr50jNXFCk9G7zr4xsNyfWo//KmzlHiumK1rsf01cXDzLIzHfjQNnKSSqVKladaEQs5pHSvjvjGHIaVyewO/fuOMIl1KIh6QWmfYD/iijMZX0ZZF0bzOcf4qqX1u4GzViTbknhLqKrtntO2ZHO9cBMjGM17G+cfL6lOUIc0WiMXVp37sdeSS0NCGq+rUIGSCepBWByCaUamOOhKCEmZNmJifQcQI0GENvIGa7mkLjwamkhOIKowLGa26jDsGoxqUHcVhtlH1WEYizUrMXYSlZZMu+owhDE8EYkbrOW09lW9VfWc8vrnGp6PXTO6YGUU9TCsNdh+N84qAtDj2mo21H3GtmqEdYWujSzjcA4uM/1FOAb7l2sMWsqBd5grsN3KzMIugtCacqM1pTCH3nc7gooWLvYPJNLUO8JZuyNAbMpnuDPtL71i8zIDADJJaeAQAyO2OYsSloaFZDnsGneG135YgQh4BjPwq5XXTlJUBoFR0oLeeKU8xA6vVufQCJjtnyCx2Yx05n4XeeoS1mSpTxGw8jMCwQJz8FKqamRuPBYGRTAW0LcPrjL4b0QerY/qGYWx+rqRfwvbyCf42zDEQUaQkdOZ0/e238jWy/3Ek9viziHPv9dOEBRScluKBFz79Fp047oyP6/VYd1szCBjPcIoQor/L8g3LOmWpQowhVIFoKMJb9UNcyNgYwmsv7NM2VOe7CFLQGMFMX6jZ2XKk512UMvpEjA7D9OIF9dro4zbLw1DyDk/vXrz7lCE2UQq7R9ltNQNFPUtESykm/CnIgNkTf4EU5p6wI9J/atKOwr2F8HTpKRgHmAmR1sK88QjVHJACrLA2aNu5KZXbzBC4x3Vivjp7hhBRXX4CROfOBA1C6m+EExngpTKRVANxg9QXAxEFlo6QgdX3m2puhPxYOMluXHgUova52RlvcQyASyYc/f8gMhv788taPVXHhQCkjSRoFS4UKRqLrxnnV6WpngI5kgd/qm+jyW/6eb/Fi2pnEORBykovBUXPRi2PqyYRZg0zDl9WEJ28PHuCEZgsX4Eo8Zu/caCHIhxO00mFMMqF/Ru5QtZiAl2NIqojRIg9sAWMtxXUCD+zRp6XLBPvjECYk7cJzDo4nXdWKvD1QwvAR/QpTlWbUs0i2fgR/xETTwJCuCy16/Z8ZdHNbVLX++xPuxHlbEuVcbkjSSLCFbhKJkjsr5xNJBhg99/dxBs9XEc5/en4VC3dCVH8DMtwjSV0zuj6rmeLTTGYUjLv5y8v/zpw489RIe7q+f3P1+Jjvr7PEOvenlx9oPx9Pb9xYG5Fzz6KpZK320C2Sgsb4x04/ftULQ0HGRR1dwjVhowcQo2E3VEDUGgA8s98EmR0miXGZY+Ms+nw18l7ZAe8xmoGBCS5D3U4ceF/CAMa0FJFfAfG7Veu4lbIkPiwqbuOirEW8ADCcjmXglNB9Qbb/VzaLLDa5jFfCVpqpaaNyXayGSn/QibPS2zGUxqmPNP0pzvqqHc8mVBmmy/LwvaIO+uDlOHvFcdmZj0NauLYoBczbOf/D1IFqNmxqozvkoTUWCl682oLCDvqjUTmN5aQiZ29Nm4vqWD+W7BF1OwRggdovBMfr3E/sabWb/1PzjGh6Q='}, 'spy42_workflow.py': {'sha256': 'e906b5f288e907c66be7e1690e88427811e0e684263de5d3ff1e9064aa3c67d4', 'zlib_base64': 'eNrVG2tz27jxu38Fyn6RWpm2FdtJlLoz7sW5ZHp5jO1rp/V5OBAJWawpkiVIO2om/727C4AEQEpW0tc0c5dQBLDYXewbyyAIfkzrt82cybqo+J1gRRUvhawrXqdFzhZFxeqlYFef/nI8ZXlRi3lR3L+CJ8abuljBrJjVMJ/xOBZShkEQ7C2qYsWiaNHUTSWiiKWrsqhgRg7rCazc29PvYvlgHpdcLrN0bn7+TRa5glTyGgcMmE/w00ySy6ZOs/ZXMy+rAtFo36zbx3+k5SLNxJ6CeZfWy2Ye8apOFzyupQGOUyK55NOT0wmTfCEi8Rl4EdcRrN/be31+fX51cR2dX8Hf7IwFwJfoeBolvOZS1CFMCtpJ7y+uz/EZ5+kJ0UrUHJ9DJC/Ye//x9cVPPWirIhGZgqUm2JBo0Idz9fHnyx8uoqu354A4zjoW08Pjk/nJ4fTkaP7i+enRy/nx4Smfx4kQz05fPn9xOuXT5PRlHB+9eH58fPj8ZBEfnZy+eMbnJ8/jkwRA/vD24v159KeLy6t3Hz8AzKO9vb1ELFgleBLhtiM8mPFsj8EffIQ5eDbqNb1NFzQQSjj20Rj+iWT6D8F+z44Op8fsN/SPWo9/Kp5Kwf7Es0ZcVFVRjYJzfTzMkMvE51iIRJJMis+liGuRMAQaBmrLSoDQ5SQ9YVbwRBI6ISFdw1mORB4XSZrfnQVNvdh/EYzHmq7HKq1FR9iEPSAmmr6OsFDNI1i0S9KsSjmiyROW5onI67MpCA+IU3Qv1vLsugIw7Lcs+CUPJqy3v94+XhaFFHT2I/xrAqSsQN+iuFiVmQA6lVxEZSUk7DFhWRHzbNOwxhvU8ZytUilhR0YT2IqvWV2ld3cCVBt4nsPQK3zKJarAAej1ki14moHySpaLB5gH7IuXbCkqQQquzxbhoU0AqtmXAO0B0BcAAkka1/hI4IOv244YBZytGjAgc0EmZcI0gAkrNILmbNPFEzzpNjLYnZ0ZLLqxYUSul6nUHKr5HeMZiswahCyVtQzZD3Q8jANDHpnSyuvzH1ldKBT1APBKgn0zCFsCGaAwRgr9loFbT3DWA1GJBiSEFrlngFQatm9j9ocCbNoDaIyiU9HGGpBZtPJAf0tYyD4peOgE1DmTIIAmV7DEyI2ndZrVraHIBBi9CAx7xCVYPznSryYs5yshje1Q9IL5+MJvAhwJbsnxcBQsvSS8E/UoUGBAtG5ux0g9V6/RvohgTHxoSuS0SIKvNmawbKT2DEEZmjn+Npqi0U1B/sGe6EPQ246Uo5igTBiNq5pcY64nAeZqGmLTroQVltiqiTxPSGG2sebGcgoT5jqA27Ej4g5vkoovamAC7pFKS6oA35GPuSda7eZveCaFJ0GXTV6nKy1DzmpQGqFlyVCoZWregFUAr5obNEJ2KeLiQahwogD7k+Y8I3EEoRIlC1zIcPyxrXKtZr5SkmpCEfaYZhnxFIGTbWZpjaZDgivP62yNNCor0u7gCG1LuyFBQWcfilwY0fDZCf9HSVoZMVC/jP8zg+bwEZh5yQ5AjeExLvJFeqe8N4pkhIHHaNzT+e481ArYo/O/22CazfVLpSar4l5ERZ6tQUw0lbTBNptxhYv2KcQDpjb5ilf3YEEeRXq3hIOOKapD403WKcFJZTPPUrmEH1zC8RH3jdGxbDmus9HrWMzrqKnjYBs7PNIQvwhsEyoguWIZaTOwO6UozMBHlhRCdshVKxA4jpEhyKvmAyBXgC0BJlQo8XHWwE6MAkvfMqIdQ8ODpuwmmCOaig8leUjQynSuYm11cLeOgnuig4AGpWWYoB8MQ7X2oIPToQAHjv69SStEG2DNWAABCoE33LUiYVvQXArG7Fdn7hkuRXxfFmle67XOIfaFi7xRt6jjPWQWEHGkKGEtFXg4arNGs8x1PxhoaY1NYEG1gpOQkKBg7D5aFBm4uQmMwKucloNVL8rIDOhzjEr0cGejsUZ8YJ1RdDU0nqhf1gSFlfUiBKBgjMLVPbBxpH6owHCiLGZU3Ks4kVbWArMRXq1hJxvKI3iZSDaLRfp5FGCCgHDrFMIBtQ7HTaIT/jUt36CYtMBA3B5B5niWFY8weHqsI1NUUkj60gdLO1BsKaYHscVIViSa2rC6y4r5KPhNMPbkT4srhdutiKI54PkaSaYwUQfjGdDzAEF0YXhIhEi9rZrqnIe3lzaIwJhG+EgYBOR6laX5/Whg6ZZMQ+EjVUiqjUCNAR6AmxcZZLsIVNoRHv4hFT8DT1aNSBg6wRqD1myiegwvy4zHYhT88guagwMPbJovCgBrHek7eEORDEgkxzwEPPPZaHo4hXz16GjCppB6HKr/xn1YIWQtogJDBia2Rod1WBwdHp4eH7Pf/Y4dnfbno/JBmCSjel0KG5N3n6Kr64+XF6+7JI/kkmKwsMzvAibA3joLXl+8+en8+uK1sw3JLAEoSpGPgmoekEjKoqlioFKLphpFnLQYg6jEAjXblmPlAfoHruoEQE25RmyK+d9GBr5aArGdyO/q5ZmVmXq62J5VT9G1+bHeaysUkWVrk/URPkzYPSSJGG2UBYWVWj4xlodh7alh4YpHOp1QZtZLyEFTuvkIUs3Cp20W9+e8gjjsLoeUGdxzL78GroLltZx0twcirPbAJ3d/IEMNwcNOCT2CkGldVOsDzLaSdLFArVMuFP0weCdgJ+DopVRJeocu+MzaXEuIcTggHx36cKojtYTQOz02Bik2iWtweDR9dnxy+vzFSyqPLAKyQjGO6ZXfVqKAmB7LYxCFpAm7enu+jyUZBSlss/22rBRDYg3G1dCgKg96EMiOqqKoWyWIHIS8WcYrea8VKyhqQ4X3F4FLD4HF6SLFuMvlZCsDtNgy6phi6JddYQXrOxU4WkrDXIR7cZyHhdnIex2qZGK09QAu2jXadrepu7bcUksXpphwsF1KwqiAxn6mHGMBVm7J/vzx8o/R5ceP10bcbE/sY0feGE2xz/KQvMFvWWBOGSKuTiQ7g7ILeee5jXG3EQZzOoHdQMIrFVlgtIfHpqNVWlNB1GUo9Iucnii22KrZXTSxVXKcKpkrC7oQ1jeuQ5x07KsvNEqTRC6xyqwLrG6+/lhU91qDIFessapdYiCrg6CofalDMZwP5uZs2lXPXhePOWYSwF9ticAmpSuslGu6qE4F6IPoHejMR5UnV3ORYF4gc17KJWaoedzVzzbhZtS4HTfxpTNLMYaMByxo5+KhmFJzW9sO4C2wo1uxaxjqSoIxcJGu9arND5hTE5/0XpvSxTcUTHSRT2nxmVM82Vo48TDxMbj1q4i0Qad8GqVEn7iC34H3wXkc6YItfO3k6RvmDUcHgSzX3Y0CeDONVhcvOEXOLmSgeKZVQIoY7FuBp5LF12pDvwriFNqXKd4SpTEWbrRQ24GwzgBssXEzATuntGcRtkjKje/Mb120ywqSxFGrlGjbEMF4STkDaqWOIdSlVhiGGClmjVxaYv2N562lqa8MfWTHTxXnZkMI6Ew8UiZT3aONBlHV5sXWGBD7LWp6O6FLpeOpNlmdVAXwLuFptsYKTgLu4RWDJOJwHzOJ3uEKp3KyuZ65hdbhckviCZ2uHqJnLZq6jQSpTqHU/RVW99qaH5WdKCpPhgTRNfJDCGGG/KGo3xRNnmxWBF01AY+a83km2B/AvV5dX55DSqOF5N2b6P27q6t3H37UVn68922njMXsYb2hYMs+1E0FoF3tzr/F9gxUib5LozfLBpWsST4xQm5FRYPDY0k4CrVz8q6kUnxDdhH44nrZA3XbSioQUSQ3PTx6GT3yatWUYSwfgm2kKqBPGdkNAauJC2xrqpCkQ5HNqp8PdQXzAbtLUiZyUXE00p1nDlRUIhKMAPBsravqYMiuvkkrCNBgtClnOqShYiFo29Hk+bNnytBaaIkhWgyGW41vd1cfYkH9pp+yryUEyCJualS5CZVWekc4b9IsiZAycwNfroPxpAcs2N9XLA4UIH1+ExwAQ1M2tR5o2TgMRAeIerL+pcDI+7TcNxF34K4GM0xHO8CHfqGyRcH3Of4J9kqWQRSVazr8KAomXvlHW4cvfbK8YsPMqzTAxlRemPXMQx8U1Qhmrtmg4sBMBeStMaBbvpnnY/vwPNsx22xvkD1OBDRzNXMAuBFUJAiRCdD37R8d7U+n6CAzDvEwfwADgQIYSRBVxZ5u3uEQD8jTRjF4lBomH4HaILCiuJ/z+B7eHE/dRV/dcljXheCYb+XYuyPVGnvFH0wQ5CQleDGtG3sulR+Tm/Xxfx1dqLyOTNf2ukhr2NolwfZQjFDVxZFumw3GcOCGrVtvJVZP3rO1h00OAliB9ST7Hm7nUH1zmO7cg/jXJk7QTmdzYDxMkRMXvGs4l9AVz9MFlqvQDQ7fTj4VPBkQVvTkXda0O+o83lycwV75naiUfLeTuoT8LYf8Gu8w2xUHVGpTjCjmWN1QTWYUOQErwFWo1GBCtKq2tIMMtaQqHmWbkdvCom502t3BlMkiexA6YPt+STH8p84gAHDTCcvEmA5sZ1KNNGmMkp+LLIqrooxyLM5mWLnF4Ycia1ZbxtGmq6K27KDNuUzpp74rBc4DD4ldwa1bYNX9eKG2tlav05f7mWbBzb3q1LjHaqlD29deB1RInU/AQuuuaqvcUcU/F49ZmouzYKh1qrsnsCuMPKFKJwAJ1Q/j7ts5SzMnx/qUmtSNyjJL6wg7uT5PlOSoH8gTmhrST9BjnAjhhvdaLUGLZcVougDclOhsbGaqtT53zB0cCKhqgsFJvfs2GL2xsL3tmrFUIw5KSasbdh/W1ju0jajCdkN4EjLkAM7cA0X0LP7d+mrkpWs0Fa2NfVPmW4DtsfX71uwQPlSpAmf2IFTHoMkhIHoE21hUaz+Kdkm3gw0C6FNvTJlatRSf1dPIVPnb0gJdmLe1ybbE4BcpLdM30W0TlUZAk28qfl6JkLSH+pb+W1U/p0nJfelU/J4osrgrBktqu6S1W9NZYszGZNYton1Ph8w5o2UHbW9M221BRQjWSCoabGqDsW7RjAsk6ixhUDGC9WJrR6XqlNKwSOZ1PmnyNkqdVbrEfsLaNuoGRQ5mAQwdKCq867c+vq2E6gtA593TvR9qlwbiU8atixoNxmzvNZzrji7ElkpBRRw3lRfS/IeqkBsqkHZb5U4FyJ304ptLj//BULprcnMCad0t2eSBzfwnGuY2F30UB60aj1KhrgeoPXiA6PQF79IUdzPQn3RrTv27Gpje65Zq7XYw4rS7mVLJ7yohHES3VUp3qxX+6/maY4swTHiH7m2fwmoIO0TM1cUhjXe527FaOMEXJ6wLLWjGKRPYecfN7aQU4GrAjXpxvz4f7SENDa6DJEr0xF0d5MQYrqjrqYDI/pO+kSPSOkECucr3yWyTIEGSTE0+VHPU16VAheolNs3wduv7dwv5Gyz1LCEJySwbPRfIdDQiqjfPaak1/HtaNrb1oX5P++j/XbDxLxbR/y0Xd9ujDY3laKvB6ZoqBw1WLw52W3F2DiJ2gLPVuQ+t/w7f+nQbq7oAUEoxb7Ce5BfIU69N1Pteo20eH77Q6tdgW9vjVWA7b9cvvqrYEcYov6Z24QjbhfGXXZgd926K3ZrsN9djWyM+UH11IduVWMtaYr+1ERsYt0b6FVtbFfe+v1bbF+vZNpH36dgo4zPXRfhSPPOdhsf5LvyfqbgfUfXb0i1M/aGNiK4Fr6S10hu4dTL13uzeUG+fTe3vUdLgFY59vIqugbOrIScGN2ft67y+VezMJfYIuqTodxZWXVF7l4J226j05Ic07Uxz82xebAiofs3OcwYsy1JIJrQ7xw/glvTdSRPj14RgMsSDyBE89ptXQpZFDrbjkUsICCCxb6GRZLoXbNRIMvhN5vZI3+DdT4FpE6+TwDgkNWZs6VOG89xKqLpvMnSIrZvkKPMiA4rmkb1zL9fNNzW1cL5pU1fxzgdbM8a7b7bcQ7E/2vrqwgAQapRmOm7+y1f3chyn6SQd66PEBMfZux+ZPsWbCyNImh2EDX2d2rt7VUFh51+UGIW94p4hRosivXObEvQ86wMjkmP9GggKdNqMX2Pscnu+I13+vfLA94nmyPaGL5echEGlOd2XKztfNum9/v/yl38CPvVUZg=='}, 'train_spy_images.py': {'sha256': '4d99920b6bc2402f114cfc4ba9dee26d815904c146b5f6436c7d4aed17812979', 'zlib_base64': 'eNrVfWtz28ix6Hf9CgRbexf0ghCllRwvbW6V1pYTV/y6tjc5ObosFEgOJUQgwAVASYyOzm+//ZgnHhS9m5yqs5VYBDDT0zPT09Pd093zzR8ON1V5OEvzQ5HfeOttfVXkPxz4vv9mlVyKYZFnW+/zx7+fHHt1maR5ml+GnrhbZ+k8rb2rIlsUm9oTN0m2Seq0yEMvyRdeltSiqocpgvDSfClKkc9FdHDw5Up4q2IhMq8Uc5HeiMrjQuv0TmSVh81F3rskT5cAwMs3q5koKy8pBcCcUYkSqq6LsgZMDlaiThZJnTz38sJbl+lcVCG0t0jnSV2U8HuBiIRY6abINivBoOZFDgVEDv9feHXhpTWgka83dUQIVpu1KG/SCj8m1bWXVl4u7uphJaoKuugVa5EP62I4z4pKeIsUuoJdp57PisX28OzLp4NfN0lep5moIu9N7S0KaDov4EdazYsbUUJhaBKGdJFsvfN3ZwBvM7+CkRXzDQGr4FstLrc4aHLgoePJolIDsYROHY+OR0P454Tahh+ncgSpG2KdIAxvOFymeZINcU4O5sVqhYWhUzUUovktYQ4FPCa1lxXYBEB6Cn0vL0Xt4dRCL5AkDg6WZbHy4ni5qTeliGOYPJwKaB36RvNfHRyod+UltF8J9TyvbtTPq6S6ytKZevxHVeTq9yqpr9TvEvAsVuqp2lbqZ52uBKOC84tPChH1HFKZfxa5LLcGsNCiKvYRW6EP9XaNIyvfn+XbEKZLlMksAxifxa8bJFzdJ6DH9dZLYCrX6tUakIQX8L/1gkF+fPNWwaM1dHBQl9vxgQf/KfyLcn5FLxgHfFTf8rzxIcpz9W25yYnSkgzbe90sCISTVRGuB1XhFfx+CzMqygNxNxfr2ntXLDaZeF/Ur4tNvjgvy6Jk1BiJifcexwxfQLPW02v7wYBVbw8O/nr+6fObD+/hhX8UjaKRf/Dm/avz/4hffnj7y7v3n+H9BdX1q2S1zkScLvzQ82nxxzg7+FQBW6nxR1IVyxjnEh9uYUEXt3EF9EgfmS7588H04MvZpz+df7HbUSU2a6u4WmxxKYB2c+sLLtk4qUt/evDp/OOHTy4sQoVWukYMyh6d6CexSp7ZD8dHAOj//nL2/subt+cIIxhFR6PQG0Wn9O+Po8HBm3dnfzp/D1i/Oz/DIcvXUVKWyTa4GEUnz06x2MnpU/ozejoFNgZkKiZQagnrs/7h2ILw+curBoDj4x+x5vHxCf857QRw/h8fz19+OX8V//387BOiec/TQ1zeHwP3qANYgJciQCYTEnMZDEIuBDwhXdB6h5L3+OlBfkEeI989hXcPMD2f/xK///DlzctzHAsuZe0t+3DWQ5wiZHhinlR15PkM5eXZ27fxz2/OPh9+/OUL/SDurqvBMpnD6kzlJoDst1jjhyFsANDPea0gQQVgiyLnPkHZyw2wzrwWsAUA+cHWUsySWQrUmcrtRPFxKADc2xMADz5F/sEA1sJCLL1NPY/z4jYYeMOfkJnzKmPi02wqwhKKU8H6nQ+iFMioKIEN0vtqLeYTvwL08kXlK9jIMOMqWYqAmPMY+Ra1A38lo1kCg09zWDPAvrgUEEE6rwf83cLlHpALrgdjG+qAdpfr0LuBTYp3gCitxaoKBg/98IMsrWrgvBtY34N2QxetBgz0aT9UIFqYFyTuNsjGQER1gSgEg8FOcJciFyArPA4Ou7wTGK2mAW2+SF24ecEEwm4LNblMuxHNRTsh4s7UroNTxOBsKqI3kiSq5EbEiH+AzHRMcELYnLa4oRsCwcYZOhaLYIMGyo1W17BmAn6oJl9KRETcwUjGxTU9DkyV2xL6FtewagONJbYbLWBvrAIzhLLtAQlkAHhyDLJhlgEnz5N88jrJKjHwvvf8/we8WEOC3bZYwIY88Tf1cvhMflF0j/CanaRuIWVfwCCF2NGps9QIN5JqqFKEMhSj32xroJopYdsHHhLTlhg0xg1mTe7XlfWWGkxAZPQ+bXJcuLSzmhEiLvNx+0XVRGqhqc8y4CCwCuorePuRpG+vZBCR90slSEgDgRR2RijouwABipgVxTUCSDwkJFgoC+9PRXGZCe9lAZKiAkZMS7YIZYFS51dtgNSzQ/oX5F/kyOskxXrettiUXnGbwwzdpGWRr5BuTG1nhlACge0jBqa1TC8DfIzLoqjHSMfefz0ybVAQNgosY2oOIlA5oHebSpTBAOawKrIbETBZcjNQx5AHATn0fBeTCL/5ZjnndSA/gBgQ+BmM5CyZX8NmPhoMvD9MvJPj5uz+FZccz63/BaeMtkuQgwA/2HRAOJdAJqAwpcTomwhEEgPisGKLpQKfdBeQg3KRxfOyWMPWAXtAlv5TkJjEykvPd4tdzIo7GAerU9DAQH+FPhPdGaYD5UMPWeYAySMTOb7hro8dymj1fum/S2HPBtGZ6IoEAs8g5SGSIAZA+w+qw/jfHUgSW/j/3RH8PQJcV8k6IC4aIvItXIOR92ICtbwXUAV/HhGzpbdbfLvlt4NH0X0JCDE+3moDqiWKAAlvPy7mMxSMK4W05CI8pDaRr6SOGiN3u9tB5OtFhPLy6zJZSWYBmtQn4ELe/AoWUpEVl1vqlFSEoR7rwaDUCtQUkWNJZWxILB+wAXLIAUff/41rZonYQBVAjhgiKGeGWanVo7oYwVeLR29QHMmqiSPfm88sat5bcj4NiSvsq1cs8dPTgwFxLcQ6hpFONlkNuwXvSHInkMRBHbiwGpnCBoRWCdTpobtJvg0Gu1bvK1Xa0zBwKeo++82mLOx/e1sGSFdjm5zZCFDhhARw1UsaJWgVaDhPQCLxht69lNRDRxoPpQQOf7P0RvhaWjOgdyC69H/Jr3Pk8gozj5quYCVXoE9Cbw2cwYPFyKgYLGkBGj/2zNUulOxoNV3cVtBH6h/szvOLRk/FrwH9HExtjoC1IrFa19tHlzsZmlQnrlBjL6APCPKBwVhMyVIoK14TdRErET3A0heO0gn6lMBmqolPLfsO27KhgTCI88UUgpwSeRpOrFNoUUc4cCj7V5sZfqZxfJynqQ5JSw13YFEoWRREDTVtDNBhxVrH7u2z0cJ395gYtQXuhTMG0PssCx7rjH8OrG7Lti2xsAEwu14WKDoapNvr0yIdpv1BVG1AfMfd7GjX6jxXa07cgUoIKilIdR5C4NXa2AmoNS0oAuukdmMcMeagnTsBcy+1mJRdCWWfKX96wn/SfJ5tFiJmG98YNqMig+kxDJAlaFzj+iOJ0m51tpA2S/RvR2jMAbrMeBywL6FXXadrso0VIIKWvGJnApVwD+16+EXSndqODggcbm308UqQ5H+H8u5VUq4yUPKtPe9QckFUcZF3KaMlqrrey89/lSoH1IENpyJ0CIeq6DBPkuKPdIh7M3RhsSkJB2k5jVRH5UzC8FeGy/K0aHpiDopczBRUGp41/P0U5crUX5ADJfM59j5FAXF+LRZGtte2dNtK60krrSNhW6tNI2ZYxp77wWAn+38jBTkaEdUKMH/d3oMtPtukSiPEbTgjt1uAluuMd2AcnU0uOcBzBaLRDiuOek1+teDDFDnplOGoMi9UtRSYQOhzRP9e8L+a26AwHejeDqYNE4RcGgDHkZaiebHeBj1DaYZM1f4eOIBj5WxWlOu9q6Jr0qQStyksvKBTyhtEaIILcnGbgVoz8YFwWmoy2p6BcYlkZRq8EtIajOa8ACCRWAnjzwXlaCRAqTlOmNyVcGwlqijS4CMDMjYXrrKTaJUiord7CRLolmtrQeVfIPRK2OaD4kuTLFnNFomH4xbzudXYfvB+8kZm74XFEtgfh6DCSKOAJr7/UZFajQ1LZDg/MXPW4MK1xlsSN60toTUgEE4tY9EOskaBEcYR++qStSsj8L4OBadaQIHRQstdYD71CybEyeIZ8FSQKyZKSjBHA0acbsoydk1e4Begsh5N95Rk3hf5LEvya0+3pTYpkmRmAtRXGIEjWxizhiT4ihOLBi4S/3xtjJBmpOT4Ad9hcmLbZWefehbae1SCEap9Qiq7BojfQyuOhPmNd44SFUik/wDWDr1AfVudi8rRAZligwL6DW7fnpytrKiqSIPJizxGEK1JbA7QNMpFMHKm0ygYCooZRaIBW0B/bGI/4HmFsLq0SraMemO+I78Hif/uwQJmbX807EEliupCAXZSZMhkVlSnHTZa5tR64gW7Rqk1yMjIANMKiQh2xrq1fNRZVAy7YpK50pBu1lDjIHx8lqzWrIpGNnpsyH5mujKdlkLLIq2Sy1II3hL1eTx3tdoh+aOFOzbMt9v8E0LNDOSwGy5En0gGx4+/1dpJUvNEb994MuC0MmhJPDwraz6biHXhumCBZ5eE9kabpJjSqhpGB8eKLNZCmTdROwNqs3Qyu8UlrPa2feR12jiGJvO8M+D4Qo63sXwGNOpjPlqP6N/QMt8Bb7yzFCxiclMadKvCWIpBC5yjK5FeXuEksLZXAZSDTlOl2whPBnqtxGx1NYSOhuUS+xXcjbwn3I46OnW/b/E7I9Bd4O7oEQBHLQDOEHKfaNw0rspWv0quRUzOOPFNKm4D6aHQGl3LyD5umOwd3ZXRGCN2MBw//PA0NMOs3v7x6TOph7amA7QzstGSKZwkJLZ6az+WbBvCR5A4YOcBcQ8PMk6OvVmCfj5AjHhKlsyvpRZKxvmFqEW5gi2rqtM5KoN5hWer0q66zNI1VC3RcwVPdWH7KsrhP9IaKuF78j7Jttx9wIwPfkkZTTz2gJH8NvJ+BiWZiRTZLSDNbhn22Q1PhaOIwkKR1PfCe3qC0Gmw6GnXsnyHkzbESVP1cbi4Lq1TkcyvUNBAhx4BzBPg6cVJS1pRO0wrDEUd+J/+9LMswOM/aS25UNLAxc7DiikDkVO3C8rOM42phUtsVqime/nqCZeI5OOhFzjP30s8IrVCOqECgSh4Q+9ZCOviLngWOqUGalFVtPz1EFFj/DYIJD9x6oWSzj8JkpvRFPH27P3L//zw2YWox0siLGFqGghcZjXctxWLIxC5TGRJUPB4zsMGaIDlgygxv/ZNrWgNJCQCp/dQb0TnZH2luCNcrHMo+QgbKkp2xLsp7O61yEH/6ODzg7E231jHs6xZosAh/WAq9oSR5NbyfAEqOT49jVgZ+wb2oTtYnAT/Pdmy6GwMrXQgj1bo1EbMA7jJEjnDwgPpgexPRUnre7kBPgQ8Cb0/HFQC/jH0HGcfbN723bEHg724kHNICYd6g9JberkpNrJf9G9EzGxdVCI4hmkA7WRAx9fzDA1m1BuUVl/x/szjhoMcxyjAx7GhLNA2l5am2W+81CrZ2LEhhpbRoX+LsLaJHVtF93Zhdj1WJ6Xi0GEctYQM7Fake9MWsdyCrvbbrdXaNhtdUZ89ywNC56PmMPzD/cjcesL9dT81OkqMz3lzYM1nJnKYTqxH22rqGEuYrvBc13RzYNcGaHg2IyHASi2qFAVkmgD3vIZsYQpIlKK0rkqbQxopnTbFY3c2cM+9dQ/UBtb8AwRe9GSKIqmQTE7FppwLV9KXPK1TkrH/47qhPWeK4U2sqQp5NiZmghxI1okRsSi1i1pMC5u3FeAvVwLtvLfASuZJlpS2uVop0eT4lqN0w3IJ+0cvRYIShqUH2yobMbOAhtHSkwYoUXSSEMglQlby8yS3FEFyresCaNwifwtYxdJoWEKDe0gNhqy3YGu2PQkYGOoOufKQMf41zNc+r7dEGST/BHkesTerRafdPK7N5xr8iKUCwc4ciw7OoideHijEyj2b+eTE84Ft/1PYrkyEDnCPYgOMjsYIio2io9OwT28lggCFT3dg0Gh72zbTGPdfpqSIqKdSDsAgEcCudvQs/hsROIq1osrxjQNJOgef0x9yZa/wXZ9VqN/HSZ96nHm3RXlNZzCWNxEdSuqXHlmFaDNfRN6ZVLAaPkkaouPOpDhiBVNVXz1Xh56b9SVo8NKrHY/x1iDJ0BleC+aAhw566Xy65XGCqWoOXfTq/PXZL2+/4GIwtMJLQPv07ZwrWkLoHDRD1CZ6KgLZ6ET+Hfyb5+ZlsckW0toM/JtEbt3dyPtoOqdGAw1duWj5i2mIoA+lZYVGqdscT1b4eGs4rFbFteATLbRwJtDoUJXx5lcC1LW9Z8YZvWg5R2kvj96ga2FabxtrhQq3FitUab1rV5OLluG/4odAvuxoRbO2GM8uuNZbGL2kDE6PjlEya9dRRtCeKj+4VXDoUAVeoTorTwisodCfqi47rv4ayaVWxbhGuobCmwAfA7XabxqVu0tmyVaUJ34HT9qNL9d7BO3dqCNjfhxJeZ5yr3mzxjnkfj7sZfZWPjGqAc/w/vtWo3jW5OxGVEJKV8ibnc2lZwPgOlh60OxlD2H/Qc7cI0wnwkCtYEB6D42J9zNy5PdoFNG6TsVmPh60qAUQkEDMyOTQt8x2UUcbKUkQ3Gu3x8r/Gcq7wwo0dpuUCzmwJIY1HWaUBKUEV7mEA6fxQFZ1m82Ky7Q2FZ0VHii4g6j6dSMEqOnDxhovEy0tOyvdVHWKr8QiTVCqg3oXYzxpcj5/430oFxRpMNuyflpu5iyk/3pEnpC/nvKfH0doCtVuGRk7cLhTqOPSkAOQwklms/ZWcSHRGnqvo6pY1utsUwUSw9F0EEq0w/YEy4rfd1Q8ng6m7RqLdDUZHoU9wratydC8hKYTB7j99oqIPXIh08yTJ9dAQZctN6uGdYGNE/OrAnTteCFu0rkw5/1a+ks2deHvsE+kS9vrRFWwNTXjkuLPN4vE147mET6iHT25SdIMA9NgAZPU4c/XG78bPINQXiu9cHY6r/svf3l1ZsHFgzqgrj99/AUFNw2lcUrCbTUHSo0jKtaVoKN/HD1UMl33era6RroMw87XUfeHrrFmDFZJvkmyuFF+16CawbCKWGBiPCk1oLgQ8hGB/sJQGnbxmcjnV6ukvFZeV/1FXeO03NOUZb66xlMGGEFF8XxgJXHE73r56qNeWcQ+gsGCLfdUeylJF7knoJiM7Fl8HfFRLBpmqyrG8KZivY1ROY+5psTsApuYKvz4SU02MJ8ZjFmMx7kBiMxoGBKL0CtmaMOHcdzVG11oz/5o+K0usYMAtKBAEpYhEd2UTKmyJmNPdTLQxDNyjNRgc3GrtHwd3ddB+avkLl0BBhLCE9k8Rkfxm6F3FI0G+sMgWokk14ymSi9XRbqQozu2op9CUKZXIB/QDmI0SgIGS8gU5KFBewOxE/iAO0FgmUYVUdmOAN6hDT/0hk9hM3k6cnoIbaGNHf98j3DF3ToYckM6eEbSDchtJYgSgXJkVEG18kQudEPqWt97w0Pkwa1j6eV3bnekod9qw63TiOhrVZVEabtSyPXl/R/nrQNoYOEYtpq310jjq0V6QOposFOtATcftWP2fAy8HKEYO59vymS+hUekZxRxi0tacebNrExFKR8f5F4JooFDHY3xOBLDP8K/SKzwyyGC+wMTf+QTCw8sfK1TSp/NgzciXqL3AQeLsrGIC0vKt6tY3eGSgYuZ9xPaUE4d/wMcItl6B0RrOBiis9sPJSqwHGEkoGwghwZD0wLsv5oHVeAISF6V0WtXCy1Ww2rUuVWoDNJiUjYIxjTgov4gV5MWH9V6Ytmucz1JRmXzjN5lJEVEZ0nwu85lJEE3l5Dm6PssHwbfXj4SCDsiAfOsJlKaVvjYzfM7s4TanFvuFEiSssW+1fPg7g4S9LhjY3C3BB1r3b0N9KwPiYxNITjjsb1JdtMotLfPptKxpXSRJQvn8SoRZNy16DOZKfJyNsUxu761V1ZerNBhOX42Ws/JTwg2V3iksGiQwLs7E6gmfpq4bYBKAaShvr5ofAW94ZFeQZ80CmS6d7oXNKA1ezia9izAJWgM1r4YWAy+YxGqXbvx4aB3Hfq+/xlUtEyo8wFnD9ZuyBtycKX8Hsa5W2dEwYwfyqXgq/ZHreY+Iht83Y7IELTcRjIby2QvoAsq0BBqbPIU1IPAER2pUMeKtcYFJhUkEHRFreEZF42f5tVmuUznKQiqsRkh6Xkqo6Auy1RysEtRrKp1AuI17CWhd4rQnh2pMakq2q0vzBFJl0ijeJArsKm3HKw9uDDbz9Tx9eQ4vjQnpOjLtOl4joyDWM0lOigwWoM+TuMOD9M8gr5QAKf2ajEDh+QN44SUZY8bkpTvLK/6qkCvY5/J1F4S8RJP0GNsza5iz0JzC+bOdOOWS8dbNXfMQQ0RNdYnHf3BHAAjqoLdMiV6CZWiwpxFSnz+L+L1nfLz75AdUclULXWEifPkAbx/ClCuaHNwhUgFUjtXyj0NiBVjqzToF7gOdvkJfaZxsWooB+QKKRlDnWaivhUiJ9BoLThqaPOA5e2VKC1PlJY4pqHDPmHmUVd0y1ME79CuMzxCNxI3zh/q3Jn1ZuHTMantyeop0zP5jgvbvKjqeLau7AO7UbibfX/8eyNtCVKjWAyVB16WbcgwSoeffMIL0vAG5oWTkVTex/ytCeXlfroEJ192kJqkfHUG3rkQLLqX3joU4+OyctmE9KaZo9iO3khUErZl1c4fYES0bnUnPW4MHSso3brY3faCIaNDv4So3zxRXddvhno6QOk8GsWjUcNIYfG/hajmZbqWSkYjAuuNnoIbNTtee9oY6HMvqSpOFKadIdCNrpko4UpQqItX0LZDkyk4A5i4S2usgTsrQY6AzqwUX0DUl5eijJog3+v0NEQRocwFhnQMUkiWkYfiao3ub5SEjc7T5sk6raF1OvW1o8UcjirT6yhmSrPqcFNVjudEFuN5aJdrCXh2QZgqGzyyLn7mEFh9Wuq0O0cBeJYmquVA09sEY/uBAAnGoAvn9aburTrcWZcnehGTR2EM07KOFcHprqkXbbGdWBTTTAxboRw7XdEhdyVaUkATD1b/cPSA5ylswKeuOeAfG2ytk3MLLc3cRZx9/ffE/kHbeHFbjzXTCfZm0buZvOTRIA0pQtUOZKPfxr5NdiiU96Y6mt3avXNM33VKGbmeUvqup6e2l9lmteLwInfL0gy7jxGHGteJJrEm1AtfF/eRY+qndkGRpZcpjBeV02/Ngp7iRm0NnPHZ0yMQJWvgYItA1mZsFFwcHnQBw8HBvzAs1thxZL6NBQ+lojVHmG6AUXUaYf4XjyyDqe0i5ITV7VVZxeI5JgOFblv7aPJ+NSvjxupqCNd/O3vzhUTpOC9i1dFYr0BLQiZUG75DPp4NgNJvcWVrBhtlzWxAMSvpmS710FQv0MvZCHUSpxAz0qhIRhjJsRfsM57Soc8i10E7FMHWVqwRVCg51Tt1FVVyH4WFy+IXXpJYnzrB238M23+MOzZn9pNds/tkM+T95mH3HCjmONuk2SKmKPXS5AuoyKTX8NoNgeN5Mzyfj9GlmphdSG5VoqzkU3UFem8mnQpA6qJjt4MdB5FyMkwGx8Dx/aVmTZsT81O3NZF/Q8yGGUt0JvKvGY51imO/KsrthLGKUCI0h5PAG6F8WtWosjeg4PpsagRkrKHUUuSAymiT2BNKvR17Ex7sdFNmTHozIbAUxT4SB1aaErk1kGuqPKZSOW3J7dQ+jkLexj4FoRfT/7SHI0VJMp7jDneD0DmbJ1ykd0JUF4FEnaIhZ5hEAEOyyYekx3VBHvAtRJ3g7Efz9Qb+bYbONV0CTNraPSpSX9FNOnST6AiK0wXiD0zfu0JMMQ+L3Hbuux3LGhHPBt4g7K5AvbYMDWSUIRynfVV+Bd1CO7uqmnocLmQHR/31T/epf9Rf/8d96h931X/g+Sg29Zoc1tg3G8b+EhRvi7wpoQtoZUU+cfKvSg4qJj7sY+jGDH98GyYG+ygJBnh/cutL3Y1sXqoMD7oVYezaqbhY80SDVduOpdppjrFjMC8u3DkLG3MQNsZ02hP7rE28gE11lRyfPm3mFdR5OxfpJToxTlTq4kiWHxjGQOGO5LLul7PuLAkU5X21ySlhH2jdZcBb7ViWpLQEwRHmc37i4R+YspnfjPJmXKLNGmcuIHjOcMvvV+KOfwUWE8UcjzHPh5y9eJGWbtAq7Drkt8mbSiNgVTr4UzSFAbA75cZSwjNyaw00CFB2pEEFhJaUisj/9u/frr5dfPn2z9+++/Zz/O3yPy0vc4kN/TkEboHNxOUGNF14XMrne2ruwVcS7ZKnihJcOr6H9AYN7cDxgfwwJIRcGXG2OJgVpgwzZZIqJEug553VrpOvSgHsCk/G+NdzQqHHVXfp32OjD16SIVlsVcY6TLVC6QbLlJIwei/JNwmda8WtNxzyrAwBz4ZvrZXEc8+En+04XEqbQo6764LyJ+qfJsRZExJvWTuEEVNb+39gA02oGJK3BplpTrLehNyeQuWbTKKfhXMCUGCe5hSeObGauPDtT3KivvHek0VOOyWv287Pt1eYrqAU+J0CTy2gkZEboDE3PuGgHVkwsXI1dYYTTGwcL/zWd1s0lv6MjSryrSo4sAQbmjyUp0WMwlRgjw1Hz5iPKNSzldgaWwajhZGBIyoxrbBEZo2QJBx27kR/u7HODB+9TzDDcTIXzaj4DlppxXIhrGi/mHk3C6fK/tOXpnRwYMfuYKSOTpQvTydJLKS0jLMtJ3GCLQ3LKQ9h2DTnYgHzwfRB72MpTDaThllhUXLCdRCt0W/2qWulfZraezgyaPZksrk/DZ8pIHn+hF4zP0O3fZOHx31vOBp9EOtifoUIHrnvjfoA347db0rWn9h5cfCDtQRtJzr9nYmPHCWV/6P+puK4uAUZYvfjUxCfjk/cVaeG1DxE5KJ70gr0a09D401XRenFY6CzJ486/WxAaLn5tHNlKdKiwHhzBsy5y+TBDgahzzCRSk5JEtVZjh4w14vVGkyNc0tVgxFu+kXimkHVD/V1TKuMloSjkMKoT0JZ5rJRZqBir7UDKBOU9qa0QjQM41SO+x0EaDZtNEIG5PyuDDlNKpJFpEe6BrbE/Ev1JscERfhFOtXKNge2CaOxqVimDOVHjgYGFc9jGw8MGrEOtlGJ81G8ChrI2obmNvcf63GyiimOP25El/lah/DHlOTXclhRRomv270auIYd+1cbv/22KWtXUSxTJmuA4fZ5Wftjd5H7tMDVW3p4OHC2VZWfZNIyrtis0yzSsLFVoJe4xqTFlHdAbwdUh02usbMti/Z0ZzJ140bbhtTVaceO0+DG2ow0sVmxldRMmnpYMuR5mSjjiZVSwD7j34lce9T+FeixNNWPHx6nrQBeqcVLehOdLZLV3wxyFyZYqRW6xCKPHatEsn93ZJIlnGUl456JpKRZwWWhhFbgwPNkOzkSwxMb3QpTsm+yFrpZGetP0cuigrV3luciwUQPbz8Fupeh9yUGZjyxtmQGLD0qiEKcLa87H5pyrrRObzVxLUtOrWNihm3g6qiI+Yq8nMllmDdo9SP2J2+MgXU8h00CDbsJIFKjrVZfnmGbdIn3k6HXZp2WrEJJo6prqFkDIUAp6+4RCxDd9BSrUBxyH+JIdLqCqmVKtt1RsDAe+7LZ2PuJzoDJkPzcSQwm7ig6dcGRjFZWM7+DPRvQgQV7aGDjoSom1+CzhL+lmGspPvvyCS+h6dgtKNcvgLvQF7cc0b/H9O8P9O/JtNtVx66L98ZQ9lAZ7i0TAvLaRoNcQ5lhDGTY/7zY5MiwG0JQq7QKM5DFlbekTVTOoa2Fa7uhplTVXa+jyUbFi87kbK0DZD3ilqOzXB0At71k7LNnWxUdO/JFSIB5G2Aw/Nve8Z2NAwo1dxKfeaE0msp91Wn+coPGA5zqG5hxWkpKfu0etVKAamWWVsPFglwT+dClspS/0PVnRNlVn/JEKN8qMVYenDZ9Iug6H/RwLdaILkZMb7Ja3uVDN6gtBJqP8VuCPmUVm0QoQQtGMep8uy3XCHkgpq9NsXSlQxDlNrlzaUSouVnTchFLG5Rbf4bLRu4dNbd9BRJYQWfF8jiBylTzgkRKvjomX3K4UwHEDTrkjVhxvtSRylsEtGozNWDBm+rKvqKFihiDo/+K98Z7poGH51qAnNzrkNXnTKuT+8ZihQ++BcoQA5dsrjaEg0ZSlRzi7YeXfzl/Ffk2lo0s/r1aJffD//zuw1/OvZd/Pn/5F+/D+7d/H8v4MGWbCb2T709kvuoSNsi5yrPloRjhNm2S1dPWSFmK6X6ro9BRYr93bpOgW8fY9RwNlasCKK/I07mlH0qrSCN6tS5qdsy696Xv44icVme0fPi39P3Wz5ZbevMoScegVpRBg7wJYro4xhUPGwlBqbbKU/LIOVJ3okasbR5sCGxKl0mLOLnSo2AZdRQT6cfvBadFH/JBIxkMkwrjSQYqwR011FGb9v6O+w7dGojPeZW2AvPM2LgV5Nyi4aUzDM60qya0iWhF4wTtfs/0aeQFDMm71RnQGHxXIthmgCDU6830+hqHHEb5IzI1k8lWJnzVli2E8dwyP8MCKTAWxs4tqtCniEeK0m4kMpHXGsoLCzGahqaOEsDFQVvkppxs9HVyGo36SKCqxbrREu3xaO8DdtU1q7xIL3iFTjFdtnFQ1gegGFhBgDqr4nq2asLjnhXV4rcqy1d7Asi5qlvAMg1NOk7NpXm2pbh1aaeWNu5wBOXlhNTpesa7uZk0QEecCp0PrdPFg3b8t/Q4Ma22gqIea9ecBIbN80dTetDRtNqdW323HfsxUszBYNdqdTtz0RED1D0EgMjCUaRIPqItCzYN+ttwCbI2Btp+GrR+aNNRX1XerFxS36ui2dna5L4LgO3SM/uHIG9DlGJxHvqLWgEGO2eqH4IVd9gBQX/dAcH0eI8pboBxjAQAwfA04oLAHIvNuroYTaEzZau2uh1y3BJRMKc+yy9NRzBLIFUeGExj1tmq7T8gy1L2ZUyb70q7etoVSMqdz64h8qa/hmTnupH45ySR3RMZj0fHiwepPtFwTe4ZtYvvHJL+bjqOTpaOjKrlVKQIt2oHtewEIOerE4D81ltf15EzgwWPlg9VhyTsLnM8KSeO88JWDigG99Tdt83nsK0tyPejjk0X9Z0OX5v28d9YirXWcWGH/8nv02P31mfbNVzG9zV8Y39bUttS5QB6CJta4MA5lHAnrDFJ39tHZca41xBhSIxz6v00kacb0Fd0Pxs3Utjh2oLllJTZliUzOjJaojGTV9cDD1nVpRtp4ipFcn1gOdi5B/WP+AC4ksO/Rx6xrApoh2zEaO4leUifpSaWrkgyh/czZMoLx+/poEvWaDtBOdaPCydGb2qbfZWvKucM6vCd3xfFsEv4aSXwN2Rs+bGy0RhfWAOvfFyNs7o8AUJvWfVuX5Ved7Pp1+64cFulpOcvFrEWbExe1bAsNhVGCMTu1c2+9OvnMeExtYzPnMI3pkXtP+ZD7MZU2oasDk/mWCOufZox4Ms+CDRz1mdF+2jKoLfPXGAkmA41wpAldCvOxHdV303Zzc3IT2vlKZ9YwQhbr1ii/UzFG9WUNXEmpO8CDBNn9umMJ9KSbV83fj0aHf7648ij4zvM+Udx4t6z0beU/PPw7MsnT8VqR95LGU6EeIKoVMEAL8KOm26tO7mfe1a3cu/Dn9++PLwCAXuYcXABXxPeY/IzwR2THppssQQZ2zNthtnttya7ou8SdQdGW5hywgKIX5LhpUekciy0lPJz4sbFaeOp2u5c4WOfrdAUBhUt1rSG5V3977epfE5smR65Xe38Pj3Tnp3uzB2oLtqH5/92hdM+nbKCFSlMgWNDxl1BsTswaHP9/XYQV2bsCFfq3QFaC01ZzC/YGs4i27ThkyffdlXolOma9bsLNcBZEgr77RIY2++3U5KzABToM20dUWJ9fUj5u04S+upZe5iqaL3aXdcaFEkvCkSLXTgZXuIMA1Yc3wjbQ0yes9tHSvqgfmSfPyz9z4DZwjITjr17A+nB7zu48N93Xu54K0DOkzc7ov0eXyO9dgmxjs85tmZ7rsJkBDghLZ9n11WVSyhHQ/m408PQzLJ7D7mE1EEHgw5JpbNiixD6TqCsOh3HT+lyJ8GjH7NBsXPJ7HJPe2kUhEW6XKJfn74BhR2qELrMkr+RvfEH/xodQ0657H9o9cNZM2Gvc2qD/B+7xrWLguwIVcbNzQ0p78KxgrfwWG9HbCoDkUyt5bvndP/r+61G2FoPoduiTIbC0Zud99w2vJsu+PLPadh5YWfYuKpyojP4Oh66+7pSZRSraO8DDROCfe3EE7ucMUlYmMoTS9s7e2AFlTW9mro8mI4MK1TuSjvdlJxhtjNc9erISjHmvndA4TsaTMJCuqJhNG1EXxvHnZY2qxNbXVA4p1RobfnMoqqGPtt4lIMIzQ9aQrcLpEOBmralcZa4ZVYcRwa/sLrmCNu6baJDFIuPQOl7efb2bfzzm7PPPqYegRcff/minkcyZtd/uGDo3LZ9QGiJ1qDl6GgtGi8nJsmRan89bRY97S36Y7Pojx1FH9TlQZusIfDvcKVyo+maly84iRHUfZ+ymHMRuK8UrPiqKNN/si5NCipKnJQfux1Da4zqOjAWfXZwZuxIW7JCOpqydoiKkxkoirF2imKJWBVshf7iDURG1rEDjo1Q25D4UQ+NqavkW9WpATjxvoQW6LabimOJF0WGuZHseq6p9VpsxzL10xN7jqlRmN+jE7xsA4/9oWRokkSZoEy8rKWyLIQPHfnYTCa2kvIsxXg2bveBSBVE2BfokP5i4nwA8rNJgTKf86Bg+BEOPqAnmTha4qXi2NFP0z94NN3C/nTcSdzsjKOW2oKJ9cFGdB/Hv04TjMONOkpMd1o+nNrNz3ZVzKCL/myU+6VtMnlPd87jMZCgCFq6t3ORpNlWbY1oKIGVgy0lmbmqEvPKEOymxWSRVpSoZauiaSpTiRPhJmz7QacskJbx1rrhLKlalzz4UtxGERUwSzyUkko8WLoUfTYWTOLQM2d2dgVkXxedvIGMfsSLW8UZmBpiLPeJfIKG8oz1NqWr32nTf063BPIdy5W+ZVaZVeAJz+YwRbrf0KcsWZqFQssWo3QpRqjH7+ot1ZLOSME9LoPvNBv9bvowoFvVqUffdQwAlHCdr9hJ/2OAt5Xmg8m9Ranj6IflQ+TdW0T/0Ot0JWVl7EKMtzQFqhcsCSvVKqa76ON/iTjcHWcq029J++9jgnMjDQJGGj393ylS77c0WgrWGcdnDSkMzWoVKAatoGreWOEicyvOnYkAwjhUpWa2gne1LovYWaYECoixloIhCt8EDGnIOqKWWAqmCLI/6tjZzptgeyNh/dfYJl8hI8PcVTwsN0f31VBeb2aaxg30uQyJbbAzXkkY97OQJ2NpXq1xAQLXSGvDLi9FvklzkW0JirzwE9eBzfWaw7tvaO033uciE2ghydI5JgybzwFtr5b2D8mQVTpPtSOs6L4rGVTIAem7wwlxqkBmAy5Y3NLEWSh8pc6FtR/Rufp1rUd1qr2zkPQmHNlD27IZ4VfpXNz3XZrb150bWtW6jg33VrPsMKHfqWFZKPUdd+zR2dZpR2fgiM0AdznRy4xc6jnuyMmJ12/KFJoqkXTY0ahKYfsPUFkQZborhLXJDgxlknE7dkZmv7WGoCNEwG1EeQ1phdvc5mqUdNTRrSYa954bosBoTBd9Jx+y2tzbJ0DaWNqtEpJsg1Ef8B45j63ftEyA48dMhD21raNRjhTYcdQaJ5iMr/NY1dR9royKM4HaKDNHI6e6oKGD7VARi1waZTvCQ6wJakeBMGxWXfc9s+omo3AfdtJ5aMTOUmahoByfpRSV+lvQ6FlxO5rWpGm13PBVJKJFvyKHit0rDFprp+nqRvd4Iim8K/BOOcYTr6XJhxh5Zd2RmdI1vaMhBjsZ9+UG4T30zONXngC2x3HHAaChvM7xJEd2qi79CK0R/T1Y6DytLW75w+BfdxLZxqHjIHKffeTRc0ibE5lwpimBtM4mbVCDr7QXkCEFw52T7ryyH0B9uRLZAlOueMhD6V6k3BJBZaCScV2IvA+gk2cJO2Y1JNOTY1DA8Q54UGdv+Y5YHd4Uea/4iksUXPEOamJ5kuGjvN2ARbdG5xuEgJlxZzJ//FNMSwSMDTPR9kdFme1JeijY+sOh5JN7OCcYfboLQONo0j6P7K3IbB92hRrFdV3X2u8MXTjbVegcpLmWmwMrlRYr8gg2WsDOWwW2pkwIcj/zenI86D19bFVysv9R8hPMIgR6qs6FciZp+CN9VMmW8Lc36SsVWGmPJ3G8KOZxPLBqRskCtn1ZJfCHQxRwhyjO03jzzatSR6Hom4/67LsHgEku5FtVUDReJpuspqfAr9bbk2PUhFUWhR5oUKINClbUeuKfq1xMeGzIeeZkxr1Do95aB4iUBUo3stpggu1sG1MgboUMjHyoA1OhhcxSK5qAD2fFnfjoziziGkbI31FVLoUhnYDtqNwzDOyRqUaBwibVeB6d7qxJitEQFaPu2k931pYqVGfV4910xOGlqA4W8KOaXEhjjSeTM3GOx6mBJ2/I2wXUJJYwY6gJ/+cC1OUk/7BmA+iZvExagbccCbqBY7KPIfLOob4C9GunSXnpD8lLXw4aSfYGj1E0Gh3thKL28CG6oEubZS+w493Tr7yAO2fwZGfVK6dhp+IPPzxCNpR1o6vmH58+21kT071047qb3FZpPjSnOUOViLUT+VE/JLNJILGBlDBEKWGIUkLvbBpxgPmSk0sePTIpe/gQs4eTbPLcG5FRGf0feHcEAbJSif+t/O6+Y4x4BFtzj3I33TaxfPnxl7AdKsti8PcntlM2+1bzTSy2ibHazJYFJhrwO6zHjK6+gYMTe92Mvc+sNgg0vE5dA2/jmkf0ItMmILUfRvQDR4AC0m/6fIpJ3glUDh9pkaR8+DI/DTtzwAwNdplUneupUXIzaoJ9hiDv1zH32cg8ZjJeOPRayUscD3280OZII6eSXr3Yne2J4mGqQwJ6qCDp2zuUpeS5Moah/pOLSyJKC9MdLt0OUq7E/Bhu7zgdsr56ookVzQ7fDoEgd+H2yOyN7UAI+qps0a1LVfpmWEoY9k3zJE3Y271Hz/YebgeztnF1W+09MWlFBFq2VLsz2lRp5eNqh6z89mQ5bohROzKl4fz01ah2Nq/Sx9uNmRSAKA6neCEvZtCMY8o0FsfISOLYV5eCUjT9/wdnQBkn'}, 'build_spy_dataset.py': {'sha256': '8f0a572d0b775ac99fa9fb4500b78fa8202bd16f2a58206c5c30f629c11d74c0', 'zlib_base64': 'eNrFO9uO27iS7/4KngwGkhJbttzuTrczGiAnlzODk0yyucxg0GsItETbimVJI0puexoN7D/sx+z7fsp+yVYVKYmy3Z3M7gIbJLFEFovFYrFuLD169OjvVZxELOSV5An7+P53Fq54UbJ4w5dCMp5GLBW7ciCFlHGWsoTPRSLZosg2LOJxsmfvfnrz4le31/ssYcSU5ftyBXBzRBvIfB9EvORSlG6+Z4OBzKoiFGwVyzIr9m4ot9CYVWVelQyBJ+MavvdpJZgG31QSKErDpIoE4yVLBIeG8ci7qjGxRVZAw3jESl4sRQm0lUK6vUePHvWI1iBYVGVViCCApeUZrJCnaVbyEhYle726rVjmvJCifgf66kdEWMYbmF+yqKxbV1yuknhev36RWVo/b3i5qp8zqajIoQ3AaxLeGyByVZVxUr/hTGpImCWJCInMetiLrEpLUdT9aVgVhUhLV62wAXtfZCHs2vssS17tRFgBn3q9N+/e/fPvz1/8k/lsMu69/vnDx0/Bp+cf/vHqE7RYyMHByIO/Vu/N88Oui4HnDcYjC4a9evPyI7Re9xj8sSTf5IkI4sjqM4skJ8CV4pvMk7jEBy6zBe6twJebOI2ym0DCZlGn2jTdrXDqpio3+uO0LHjE90EhYKWp0TPPon3Ay6IeTbOFSSZFMzf0epPmTWz4pfky9g4mznJhTrCKlyvjNclujLdmIv2+zZJq06wEpwrKrApXghhE7zQkgLOkEBEFB0DQ0IHqkLfk+TEXFHjd3Jv13n/4+cWr4MWHd+9xs0buaPK0z0au9/Qcf668Mf5cXJ3Per++e/P57QnQp5OxAXqJoL1eJBasEDwK9OmzcbedKRF4E5crEmzV6CIfbZGGWRSnS9+qysXgciDjJZCdipskToVvWQ6eqoVCgH8KEWZFJIESCZIsIhsOovsyDssPMKso7IXTZ2ux9xO+mUecFVNWXFsvUXpmDiEpshscfT2jN1IGiEyUtuoXl/Aqxh78D3IB//+SpYJ6ysIYiFql4Dege2qSWhoRKUBCdz110xUvVC8Mo6nbQUQbj6Vgv/KkEq+KIoPVWC8rOCYhDqFxt/j/neV05pIujyIbn9p2WCaQcKvmnxLYndnpVjm22bfrKVskGS9tJHc9c2hlayTQtt5pSf9Ji/gbJZIvSKidu85s19avSrZnMC8cR8LXtLWgWZ+t+izpsxDgbBy4nn37nI7JSdDSjCeJjfrUjeUiTmNY0dYh27RlP7IRId4S4nZex/kq139OtzyJI5YXMSjKY67D5Cv2AyjyHeINAa/DYKYE5tzEqW5bUVuXNT/4bPTNs5P9PJ48L8Q2ziqpDjUKGoj09cCbXWsuzZA+knMwyIKFzUiSZyR6NQBG8LmEhy42RzUnh82OgQPELYdNiuyy6HAkgeMMvQ7zfeZNuqtUR0lWGwUxBICmXyTm4B/vGWt7Z+wxPT8BGg5R4KENkQh4iiWdWbV4e/w4hBFPH4tLHHPVDqEjrsbA46lB49Fj6MJh4/Gps/Pq7fNLX1z2GTyMPR9g++z5pw/exOcma3AnapbBs1ZDpIqpT6tNsocBCGygdLY60IoVe8ELfayw9Xo60ScKqCffBuSKYOAX3iemvqR5LDCPcWrVYxQ+gj0/hiXpI//n1ICLE8iFLBXoCWn+nMoqz0lZmy5YK9dq+Ru+FoFyF5BLEgQRDTC4A2LXZ42ZhzcfN0ozBmHqE9DCK31LPoRqQGXe9LIBa3ydJ8yrl2iC/wCqA84ujrm2aEvxUGkROeroHuojFsB5ltViEYcxuGK1W6rPs2aIXoG5SjrEZgPOjxqvFdPGMpHo4Pk6FCH10xjAFqVCYCXxVuiNE7JKEEcEttRF/xGsqLSVPwf6V9Pb+HPo+EFMAF65BUxU/FCzuIXIEx4K2xpY7UCFvz45DbMafH7z1G91SuMv+gvlPMrhLa3xbghzY0hAu3pL0z/C6R/N7tw8XVotEoL36X8tUkiBb1LcZ6bX6dNWGNIwa+AapK0TqRFpxatnIIfS7wjJwVj09DQAapFmJDl3bQf4nnpkc9zV/k0NdFJgWGBKit8Rd5BxsohdYQBFa/KgwYf+MmxuDVxblEHTQgZ6ZuiAkxvbCrfid3fy/im4Kgd/D5eoSIBDpaQUtR41gUF3To488P79A+KHXdIH3kkkdaDg48Pwns0z4NF99buYGz2FMUHTR57MaRzgvTdg6Oi0GJRwHSzkJA4VUjSgtavRhTUjDR8Z2pkVNZgpjPjepd45gc4IQTooa5H5oYPzGEMb1nyFIjwF30JSNyz6Gk2I9bQ4tRHUwf4OO4f9HjkyQ60jSbwXQdcroDNlWkBtJFNwR0FLidyWOU9BWOJwLf2n2hgWWbVcaQ8P+4fU3WeeGFwo/Hl2I9CR8Ebs8WPKQ7jg/IONosckW3ojm7Boh0+GPBERmSdoHNJw6ghBocR18GSD3wNB4NiF2BH+eqPOYtAhbsE70dl2St7mdqDmcRzw8NQcvV7w4d1vH+vwK3j3+VP9rDiBvn5wkxVriPmUu6ByRZoVyySb84QRkj7D4dRsvGub21cRqR5MQJl0RbqNiyx1ISyEyThshm29ff8GXYZXv7xEs/Z8uWxclwI8O1HUzkvJ5VpT0eZ78iQrMRuEre2rW0kwkwrVSXg33+MTxsB5UqrAE1M7JgQv4Qg1qZ0PIix5ukzESWAUCBAADfs63onoTQbRZVb02esqDV9nBUCXAl7fwqJjWI/uN6y/KZdkKeRa8U0l63zF3qGGvjbTPjOdCEBri4C4HYbfNqj9sifetG194s16RoihRjvsb37rxoFLpppVGKQtHEDUNLQ5ptlDntpvirJNLDfIVr0t82WfLQvwTFgpdnAsN1VJh8L6zjvzno7HKA/fjSfjq7MLeoy8aBKF9Hh1dnXFL5WLVeV9Btjh2Cr1mSXIdq238IVQji/4xRWnwWJxfnY+osfFYn42Uo+T8DJaLBRKEAq3CN/zgm9kE9E367MWGeb++CZO9taUWS/FF/5rxT7yVCIm6pXxn5gf8EatKrL4TkgYFgqiCnqRAdYiXlaFOG5Hnrh1CzHoABMlhmsA4h2M2qEouoet+xOtB9hEtGzmV5tiSb4VQN0haTRQJymgG7iL7FLLsOEHl+7b3tiFCO6pO4bQN8pj3zvX6oujcCNaHkUBTm23ua+zEaXJLi/o53w80uEY394/xuuM8c7PwNjLFS8goOE7p0kq8R34+Ziw4BD28K2Rq8Ae1EgBPpCJsz8VlZHzIQDkiY1PvjXPKNNKLPEVrzCrdhNH4FED2ecHQ/fNDGQ4gjyTMUaBtlWAxS2te8Fph78GjjhBC6CodumDQ71Egvos55EPm6FuEnB3rlocyBsZR0JliRKxUOnhLLcO0jmKqhwWCl48DJgRidtYxnPQzq85uJQHlGlgTbaCbxhlj9yL83sGwALKbPPACCQathGTcUC2SMEzK/CMah3WEl6f/yrXuZvWZfnRV+/KBVEuMaoRgyR3i5NLW03VuJL0rL1PLQVa6ZhicAny+GdWgAXzJ85hGIDmuUPNwCSmk/6pXfTD3aCzQFbKboyTbe8GcIrAb0DvwMDY76zdcSgLDb4FIu96W6f/NCqgXmmjLk4t/bxd+rmR4kIWH61ipVjcobVexO6J+r1nIs/1jHnafdu6c140m1Y77kNPXGBEqkjExXfQ8iRfceh4Ou6spZngrNZeLhp9uwB+C7s2lMDP66Lx8imHjYKpxLEREsM8dbhgrgjvCWSWxFEQ8lyW+0T4FjiJKV5RaEouvo0S8u4fIKW2jv8XtHzHnqN6TeJNXEoWCczJMbELk0rGW5HsWZayciXYZMyyOcTVW7DzsEfqthM64gJ0lCwHWQqwQLt0VXoOpBj9bpBlkuf2BLbhfbtY53C1jkZCFgedd/xXtEf3r2ABFYrXKeiVI8IBkuY8BiN0cV7vBiqrPbDAxr6BHtCn+Z/oN8eE3SEsSDqyud4+eB2fN1CtJdjwL1kBYS35jPaBD2m34UtLm3MvlkXtjNod19TWwcMe/PgpW1i3+6k7XtxZLaJjS7OztFVRGts3rAAQokNozfzCSNcfshjPJo1SIyB+zdUojeIxHPWJCYCLBYh24e3AOnqrD+y23Rg4zS1c030fl0033r42Jn4c0wJiuqSi0wd66JKygF0oiK3b+WbOvTN++44s797WG1K7BXR5BisD5YZXhh4Y+vGkz86g6QzaJtrL13wg/1DazWDnqFN5CfZ1VLro91LOMgY3n6jSFvY6bqIBx5VlscDbc9v6PmLfzy2n5U4zjTZpCVYR+EZM0Tt58XIUvxh3MCpMXiHbARIRtnF/F5HCHZYUl48eqzHDEzCIJOh6CnqCH5tkWeMZoOHdkdGyO5PXmrVFpuyI0psoejbszhmaXtNUotdam57LRq16zZnTNRPgWFu3NN8jmu/RTB1OsJF7H8WvQ4uDzWGGt6e+Te49RKycShrQucN6D8s5Nvu7PYYZvg1SM3JUUKaRWNliAQIC+xmnJQY5QG/tK27hMRRYHmEdo1Q8sW5WMZUhYGikHFBgVxLnQZYqnXE8cj7Pdv4tuII74qA1teQfFXj1fXRmQUtSLYERmZh8t4xgpm3XIQtloDEh/5B8j2bd2220Z/cNUaw3oREoKLSI0nyHh4R9/7vlPLHYf/3bvzPrCWA/CVCHWC5uhd1EPFfnIEl4S2By1LtSLzcCvW0MApLoYHfq3fhuMVqMFxfH+HUopfB7Lxn7z/9g7JffP75iz4uQd2Ybn0SNeI6w6koGwtpyxkQ2wgwXipSKFE6ipqD1Xo6M4Oc+LkIAn4MydUzxwwkNvJ0ZdbZqleBNuoUBsvsF5F6dQNCNd0ydxLXY61NICo/6KA2Hmu/att5ZfeXROn3b+gleyO/Alzfwgp4MPr+AZ30JPzvekPPzZnlIUIdppzTO0TJO7AMig7UoFTd9gitgzL4FTalevnesr85zz2bdtz+XFxOcE1wsdgkAWpOh89UqsgNxOHKZH5zHu7o8mGfsdSYae1+fqfGIv4WNNJX1+ePLQ7VmStVfZNP46SWSr9yklnz1/ogiGFrD29MbdJ8oH080msAgC7xxVVCIaeREyIdW8g0cUUg/UgHhlH0ss3D9L1VWCjZkv/NVlrHXccrTsGMHLo8O4ldZprNSOrXcJJf6RoyK+bM8ToL1DS+WEqxImG3A7ksZJALCEWs6uYMIWZQcLaFvJPY+xSWaGlKu7JA9qtCKZJhKE1DEqCLGrD0jNC+FDIs4J3uLjttzybJFvZv1dSr7IHjShkOqqJP9kjFVVKhnZXIFnodrdbJumG8j9wXTbZ1LgVuL8sHTk0li2J05mHgJ3Yp5oC7BfqEHF+Bu3DUlA6DtdBKFijPRL6oLNd3nxRKWm5bvqcd2DDCVntP9tlUXn8JOFuKPKi5E5GNmDVyLfS58vCB4cLQi8n86Wl1ioFDrqwZ/ooeCD/PgSHIRxQ1FozBcOU6+hfUF4MkDCdbDi17H+UCAh1/GeGN+/3gUTsyeKjT0g4ik3Xa7eqM26ygubACBOaRmA00RZGu/zVbqIrxOpSChURtR10bI9UHFnXnLfSKthmiNpBr4x50y1h90Jk1fD8CrUck6PbjLg7nrop3j0hSTjoHXfXeaq/q2Lqu9kDg54/FFhNJOKg6hapkoXixAStSNjtjlIsQ7CC4HcGBlynM4fqVl8O1+2vEOhRg18Jw2S6lKSCi3qkqF+p06oL4u8unrmhGDy7ax/0NdpmENCZ/zrfJAGquLaMPTeAFTYiW4pYtFrZsHa0NvCvDeUQfUNaG/UYO9AJ0biyRK+UZIX5W0OAejXPpZqSLS053INLrdA69bS6OkokViuBoTYgG21GU0ti7Htkvy01XBsx7ajtPePm14ANxSOqxlhNZNB1Casyd4qcrF3TDL92PzUPWN0UM0HAFZjQB5H2DFfABWaFPliuF6Nam6ODFMjy7CD5CZEOwoAzTQRoB2HxcoQk5qhWlw0+psQY7J4lieS1dacr+BKEAhI0kDxxjiQMLv0W1rkmXrOQ/X1rROQBn4avFX93vTth6dLubIG+RbWCqfJ3AU1NcKgKg9iAauRVwAuFl0PsWMntpAs/lwG6k05xjKiF61iVMXbtfe1XjU90aXI6NQw6JyUzCCqUiCsMjyIMXILYERkTVtK7ZNXqq0zT1DjNJtY8wmiwA4TtFmTa2facvyeIcfb6AlcdkbjBCGtd+Bbk+1SSVIpaACNFK3cQhOAWFiC8Hp6wLX3GRwUwN1sQNm/PqyP/ZmqnY90PI81dGEblWBuzUdHeCQAhdivcZ9qT/6IJfiGUoniFiVVRJLsKsCU7hMbrKsXKHkzQXKYX3ZDB6gzix0yFSl/1MLjw+r4z2ay5sMtqiQwf/ilBRO2W9xAtpBp0G8oQd2ep2C14MfnmDWWEm/0skUf7PPEpR0a7ionoKSrcjHB7inBGHOZQzcs96BnwkuaVKvH10wlz2PvjBiIRYGoteo/DMQSA02r0qap6qJUN/v1F4iNFAG25xXFRDO9zDp8YcXBoTuhHA1ziIg8VYbjem1+aFIH18mA288OMM6NtOeKMBzE/C8BSRTo0AuahDjvM7ujghSeteaql+jO8q+xMEKl4xOztR6tQMpYH+CMmdoUgZlNlDZvHkWxcCVFXjtbJ7wdN2WpT1Dr5VTpT85r3oz0afeoAmYw+YU4H6DqpHxAqv16XuoTHaZW3NUUKE6ZhuBcR2vw/i0BTSfKTiKxh/rVyT9mS5vV10/6DZFOrYLWimuf4gm2z3Iet335QycBXPKoTklGzDvPjzNdzZTy+5gGJgYsJhaHRB9mKjC7ggnnvz6k5cphCU4gIxRsQFZxpgmwc2KU7oRbqqK98D0mwEW36mz7LKXMV+mGZiikHSbOnh5kQE++AsHW9D3T2ga4NwnybAQX8SRmmhoMr+6OWAUHEJqJ03Ajyl+xqTAOwnw5ciH08vDtFtZdQSlI93KZFcFmsdVWeZyOhze3NxAKARh6x8YtrpxNqQ3iUWxw7q++BiJXPHx+YU11V+muerddBFccsgp9rIdx12JXRSDNwehl4GN8kciCrSnN8Wi/q36gqNPn1qoM+iCx7QBNGgY13/ztdvYwYPHPKjSQ4R6PNYvq1F9s+wTlANWL/EkKCrMBUyvu9v0iocrrYnB/5W1GcD7vloJKxcANGiSMEw617ctqtxNXfaB1YjpRkWi/mzD3/qrQrKUhyLyM9pUbXEk6IY92SjUHIIXILHFgOroaxwcDLaU2vWe44UlL0Csn4EIZc2c6jtEMr48wf3ZK6NzNPkvAhwrlPso2yR70FUgbSXpGRLJcFVkaZZkS7LcNKd02cdVtVgkZC51HRm6g+jUoD7HdpBrnoBci+hoxk+wf6ii608/kUbSWSs8TGiwEhBLYJpAS6q0IvLtGfjwSCsyFmA3apCesE+82sO5yPIcCSCEYN5WWQKnRSTHx9N0oST4vehcvAcpF8xj21hWQEjnS1VwgBWSIdUsGO6qy15ksPHYB4Nq1YjHdwABTIFU4Qle7gEHzASL2NdBv2zBYXFp2dm+Ji/SO46YaodaOdsufigK8Q5FHAElrrDFjapNLm0F00fphCn8sfPE+te0jvILLHg1gG+bGCpAbKUA5U6f2VDY0bcObKal0xhTWKNJoXOHGemkkiszyKB9oCq4a8KHcS59skAjdWYioHtyupcioDowVipnHedBnX5og7gO5tbJbpr1d18dHirH20whOS5hBv1jfAtCSDG4cDdwFMosjUMd6pVZCQqFFB+AjDp06vwMfdVk5BaO6k9Nlh1Ux0u6JM7tTqFov1lTt8gp7dcfY3RzGwpTn3kH9U4m7U98PfZaZ9BmHVDkHfsebxlVeT3Ohk8oFQ0x06MrrhOSpVaC1jkFwVkowWpQ9C2RQEwGOl0KEFn0EKkewz5k/kDvS987KWXdQhz1/ejxp8tYLqG3QvrmlvVpl2KKhgrf2DHVQTlXtXmHIu9QaiGHSbrceGB/ENh9YJP7oIGrdE3Z5AvneBv/4lb+L7bz/3VLT0yMSe9ElDgzJYisFDRH7QwcEKHzwgajDifp9YAtAaUnggC5YQUBJoqDwJrqgg/MGvf+G173N2I='}, 'SPY_daily_with_2019_warmup.csv': {'sha256': '4e2045b50251b87619b406abcdee3697862a2d69cc18744075fc15683ab57c5d', 'zlib_base64': '...'}}
BOOTSTRAP_DIR = WORK_ROOT / "runtime_code"
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)
for filename, packed in BUNDLED_FILES.items():
    if Path(filename).name != filename:
        raise ValueError("Unexpected bundled filename.")
    content = zlib.decompress(base64.b64decode(packed["zlib_base64"]))
    if hashlib.sha256(content).hexdigest() != packed["sha256"]:
        raise ValueError("Bundled file checksum mismatch: " + filename)
    (BOOTSTRAP_DIR / filename).write_bytes(content)
if str(BOOTSTRAP_DIR) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_DIR))
for module_name in ["github_artifacts", "spy42_workflow"]:
    sys.modules.pop(module_name, None)
from github_artifacts import GitHubArtifacts, file_sha256
import spy42_workflow as workflow

TRAINER_PATH = BOOTSTRAP_DIR / "train_spy_images.py"
TRAINER_SHA256 = file_sha256(TRAINER_PATH)
spec = importlib.util.spec_from_file_location("spy42_trainer_runtime", TRAINER_PATH)
trainer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = trainer
spec.loader.exec_module(trainer)
print("Loaded verified source snapshot, chart renderer, trainer, and GitHub helpers.")


## Connect to the repository

Authentication, rate-limit, and network errors stop the operation. They are never
interpreted as “model missing,” so a failed request cannot accidentally trigger training.
The notebook uses exact dataset/model release tags, not the repository's latest release.


In [ ]:
def connect_github():
    from google.colab import userdata
    try:
        token = userdata.get(GITHUB_SECRET_NAME)
    except userdata.SecretNotFoundError:
        token = None
    except userdata.NotebookAccessError:
        raise RuntimeError("Enable notebook access to " + GITHUB_SECRET_NAME + " in Colab Secrets.") from None
    except userdata.TimeoutException:
        raise RuntimeError("Reading Colab Secrets timed out. Retry this cell from the Colab interface.") from None
    return GitHubArtifacts(GITHUB_REPO, token=token)

github = connect_github()
repository = github.get_repo()
LOCAL_RUN_DIR = WORK_ROOT / "local_runs" / MODEL_TAG
preflight_release, preflight_remote_ready = workflow.inspect_model_release(github, MODEL_TAG, LOCAL_RUN_DIR)
preflight_local_ready = False if preflight_remote_ready else workflow.is_completed_run(LOCAL_RUN_DIR)
preflight_action = workflow.choose_mode(MODE, preflight_remote_ready, preflight_local_ready)
if preflight_action in {"train", "reuse_local"} or (preflight_release and preflight_release.get("draft")):
    github.require_write_access()
print("Repository:", repository["full_name"])
print("Expected action:", preflight_action)
print("No credentials are stored in files or printed.")


## Obtain the images automatically

On the first run, the notebook renders the same 1,733 charts from the embedded source,
creates a ZIP, and publishes it under the dataset tag. Later runs download that ZIP
directly from GitHub. Downloads and cached archives are checked before extraction.
An upload failure preserves the completed local files so its save step can be retried.


In [ ]:
DATA_ROOT = workflow.ensure_dataset(
    github, DATASET_TAG, WORK_ROOT, BOOTSTRAP_DIR,
    allow_bootstrap=BOOTSTRAP_DATASET_IF_MISSING, workers=RENDER_WORKERS,
)
DATASET_CONFIG = trainer.load_dataset_config(DATA_ROOT)
manifest_index = trainer.load_manifest_index(DATA_ROOT)  # Metadata only; no outcome columns.
TRAINING_FINGERPRINT = workflow.training_fingerprint(DATA_ROOT)  # Train/validation only.
display(manifest_index.groupby("split").size().rename("images").to_frame())
print("Dataset:", DATA_ROOT)
print("Training-data fingerprint:", TRAINING_FINGERPRINT[:16])
print("2026 outcome labels remain locked for model development.")


In [ ]:
# Preview the last training chart and the exact visual input used by the network.
preview_row = manifest_index.loc[manifest_index["split"].eq("train")].iloc[-1]
with Image.open(trainer.safe_image_path(DATA_ROOT, preview_row["image_path"])) as source:
    original = source.convert("RGB")
    model_view = trainer.make_model_view(original, DATASET_CONFIG, height=336, width=768)
fig, axes = plt.subplots(2, 1, figsize=(16, 11))
axes[0].imshow(original)
axes[0].set_title("42 candles · EMA 8 · EMA 21 · volume")
axes[1].imshow(model_view)
axes[1].set_title("Image-only model input: chart panels, without printed date/price labels")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Load the saved model, or train once and save it

The default model uses a pretrained ResNet18 image backbone with small direction and
ordered candle-body quantile heads. Numeric columns provide automatic labels and
reporting context, not model input features. Exact dojis are retained for body targets
and excluded from the binary direction loss.

**AUTO checks for the model every time this cell runs.** Existing completed local
training can also be saved again after a failed upload without retraining. Incomplete
training is preserved and requires a fresh model tag/work folder to restart; optimizer
state resume is not implemented. Published model versions are never overwritten.

A new version uses 2020–2024 for weights and 2025 for checkpoint/calibration/threshold
selection. The 2026 final test stays separate and disabled. A matching saved model is
reused even if you change `EPOCHS`; choose a new `MODEL_TAG` for a new experiment.


In [ ]:
# Recompute the decision here so retrying a failed save cannot restart completed training.
remote_release, remote_ready = workflow.inspect_model_release(github, MODEL_TAG, LOCAL_RUN_DIR)
local_ready = False if remote_ready else workflow.is_completed_run(LOCAL_RUN_DIR)
action = workflow.choose_mode(MODE, remote_ready, local_ready)
expected_context = {
    "repo": GITHUB_REPO, "model_tag": MODEL_TAG,
    "training_data_fingerprint": TRAINING_FINGERPRINT,
    "trainer_sha256": TRAINER_SHA256,
}
context_path = WORK_ROOT / "local_run_contexts" / (MODEL_TAG + ".json")

if action == "load_remote":
    RUN_DIR = workflow.download_model(
        github, remote_release, MODEL_TAG, WORK_ROOT,
        TRAINING_FINGERPRINT, TRAINER_SHA256,
    )
    print("Loaded saved model. Training skipped.")
else:
    github.require_write_access()
    RUN_DIR = LOCAL_RUN_DIR
    if action == "train":
        if (RUN_DIR / "run_config.json").exists():
            raise RuntimeError("An incomplete training run is preserved here. Choose a new MODEL_TAG or WORK_ROOT to restart.")
        if context_path.exists() and workflow.read_json(context_path) != expected_context:
            raise RuntimeError("This local model tag belongs to a different dataset/trainer. Choose a new tag.")
        context_path.parent.mkdir(parents=True, exist_ok=True)
        workflow.write_json(context_path, expected_context)
        if RUN_SMOKE_TEST_BEFORE_NEW_TRAINING:
            subprocess.run([
                sys.executable, str(TRAINER_PATH), "--data-root", str(DATA_ROOT),
                "--output-dir", str(RUN_DIR), "--smoke-test", "--workers", "0",
            ], check=True)
        command = [
            sys.executable, str(TRAINER_PATH), "--data-root", str(DATA_ROOT),
            "--output-dir", str(RUN_DIR), "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE), "--workers", str(NUM_WORKERS),
            "--min-validation-signals", str(MIN_VALIDATION_SIGNALS),
            "--proxy-cost-bps", str(PROXY_ROUND_TRIP_COST_BPS),
            "--pretrained" if USE_PRETRAINED_WEIGHTS else "--no-pretrained",
        ]
        if FINE_TUNE_LAYER4:
            command.append("--fine-tune-layer4")
        subprocess.run(command, check=True)
    if not context_path.is_file() or workflow.read_json(context_path) != expected_context:
        raise RuntimeError("Completed local model has no matching dataset/trainer context. It will not be relabeled or published.")
    saved_release = workflow.publish_model(
        github, MODEL_TAG, RUN_DIR, WORK_ROOT, TRAINING_FINGERPRINT,
        TRAINER_SHA256, DATASET_TAG,
    )
    print("Model saved:", saved_release["html_url"])

print("Active model folder:", RUN_DIR)


## Predict with the saved weights

This cell actually loads the completed model and runs inference. It does not train.
The latest input is the fixed snapshot's chart ending November 20, 2026. New-day
predictions require a new chart/manifest entry in a compatible dataset; the model
weights can still be reused.

`CALL_BIAS` / `PUT_BIAS` / `WAIT` are directional candidates for next close versus next
open. They do not select an option contract or an intraday EMA entry. The body interval
is nominal and does not predict the entire high-low path.


In [ ]:
latest = trainer.predict_latest(DATA_ROOT, RUN_DIR, workers=0)
display(pd.DataFrame([{
    "as_of": latest["asof_date"],
    "candidate": latest["direction_candidate"],
    "model_P_next_green": latest["model_probability_next_close_above_next_open"],
    "q10_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q10"],
    "median_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q50"],
    "q90_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q90"],
}]))
print(latest["probability_notice"])
print(latest["entry_trigger"])
print("2025 development metrics:")
print((RUN_DIR / "validation_metrics.json").read_text())


## Reserved final test: deliberately disabled

Leave `RUN_FINAL_TEST = False` during development. Once all settings are fixed,
enable it in the configuration cell and run this cell explicitly. This is the only
notebook operation that evaluates 2026 outcomes. After viewing those results,
tuning from them means 2026 is no longer an untouched assessment.

The model has already been saved to GitHub. The optional final-test report is created
locally and can be downloaded in the final cell; it is not appended to an immutable
model release. If an existing local final-test report is found, this cell displays it
instead of overwriting or rerunning it.


In [ ]:
if RUN_FINAL_TEST:
    report_path = RUN_DIR / "final_test" / "test_metrics.json"
    if not report_path.is_file():
        subprocess.run([
            sys.executable, str(TRAINER_PATH), "--data-root", str(DATA_ROOT),
            "--run-dir", str(RUN_DIR), "--final-test", "--batch-size", str(BATCH_SIZE),
            "--workers", str(NUM_WORKERS),
        ], check=True)
    print(report_path.read_text())
else:
    print("Final test LOCKED: 2026 outcome labels were not evaluated.")


## Optional local backup

Your completed model, calibration, settings, training history, and validation results
are already in the selected GitHub model release. This optional download also captures
any final-test report or latest inference created in this Colab session. Colab runtime
storage is temporary; download local-only reports before ending the session.


In [ ]:
DOWNLOAD_LOCAL_BACKUP = False
if DOWNLOAD_LOCAL_BACKUP:
    backup = WORK_ROOT / (MODEL_TAG + "_session_backup.zip")
    workflow.deterministic_zip(RUN_DIR, backup, "model_run", exclude_parts=("smoke_runs", "__pycache__"))
    from google.colab import files
    files.download(str(backup))
else:
    print("Saved model release:", f"https://github.com/{GITHUB_REPO}/releases/tag/{MODEL_TAG}")
    print("Dataset release:", f"https://github.com/{GITHUB_REPO}/releases/tag/{DATASET_TAG}")


## Troubleshooting and interpretation

- **No secret / permission error:** configure `GITHUB_TOKEN` in Colab Secrets with access
  to SPY and Contents read/write. Public downloads need no secret after setup.
- **401/403 or network error:** fix the connection/permission problem and rerun that cell.
  The notebook does not turn these errors into retraining requests.
- **Upload failed after training:** keep the runtime alive and rerun the model cell.
  It recognizes the completed local run and retries saving it. A conflicting unfinished
  draft is preserved for inspection, rather than silently deleting an asset.
- **Train a new version:** set a new `MODEL_TAG`, for example `spy42-model-v2`, before
  running. Existing published models are never replaced.
- **Model compatibility error:** use the matching dataset/model/trainer version. The
  compatibility fingerprint covers training/validation examples and image settings;
  appending new inference observations alone does not change it.
- **Probability meaning:** the direction target is next close above next open. Threshold
  selection uses 2025 and a simple signed-SPY proxy with the configured cost assumption;
  the proxy is not options profit. WAIT remains possible.

Original data: [StockQuote SPY history](https://www.stockquote.io/stocks/SPY/history),
which identifies Yahoo Finance as its upstream provider. Indicators use only known
history, and images never include the next candle. No manual pattern/trade labels are
required. Model usefulness must be established by the reserved-period evaluation.
